# DPO-family query-rewriter training

Generated by `notebooks/build_train_dpo.py`. Do not hand-edit this notebook. Change the
repository source or `configs/train_dpo.yaml`, then rebuild it.

The same trainer runs standard DPO, WPO, and robust DPO. Kaggle uses one QLoRA replica per
T4 through native TRL/Accelerate DDP. The target global batch stays 8. The retriever is not
part of this stage.

Run order — one non-interactive pass, top to bottom:

1. Package setup, source materialization, runtime config.
2. Health and exact data preflight, once, on the first arm.
3. Two-step DDP smoke, once, on the first arm.
4. Full three-epoch training for every arm in `ARMS`, each isolated from the others.
5. Deterministic Qwen outputs plus the cached temperature-zero Grok baseline.
6. Gemini tournament, which runs only when every arm in `ARMS` has a promoted adapter.
7. Export, status table, and `run_status.json`.

Set `SEED`, and `WINNING_VARIANT` for replication seeds. Nothing else needs touching
between runs: this notebook is built for Save & Run All. A failing arm is recorded and
skipped rather than aborting the arms that already finished.


In [ ]:
!pip install -q "unsloth==2026.8.22" "trl==0.24.0" "transformers==4.57.6" "datasets==4.3.0" "peft==0.18.0" "accelerate==1.14.0" "google-genai==2.20.0"
!pip uninstall -q -y torchao

In [ ]:
import os
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
RUNTIME = "kaggle"  # "kaggle" | "colab" | "local"
ARMS = ["dpo", "wpo", "robust_dpo"]  # trained in order within a single batch run
SEED = 42
WINNING_VARIANT = "wpo"  # read only when SEED != 42: the seed-42 tournament winner
if SEED != 42:
    ARMS = ["dpo", WINNING_VARIANT]
GDRIVE_BASE = "/content/drive/MyDrive/simurgh-data"

if RUNTIME == "kaggle":
    from kaggle_secrets import UserSecretsClient

    DATA_ROOT = Path("/kaggle/input/datasets/alirezahsn/simurgh-data")
    OUTPUT_ROOT = Path("/kaggle/working/dpo")
    WORKDIR = Path("/kaggle/working/dpo_job")
    secrets = UserSecretsClient()
    for key in (
        "REWRITER_BASE_URL",
        "REWRITER_API_KEY",
        "DPO_EVAL_BASE_URL",
        "DPO_EVAL_API_KEY",
        "DPO_EVAL_MODEL",
    ):
        try:
            os.environ[key] = secrets.get_secret(key)
        except Exception:
            pass
elif RUNTIME == "colab":
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    DATA_ROOT = Path(GDRIVE_BASE)
    OUTPUT_ROOT = DATA_ROOT / "dpo"
    WORKDIR = Path("/content/dpo_job")
    for key in (
        "REWRITER_BASE_URL",
        "REWRITER_API_KEY",
        "DPO_EVAL_BASE_URL",
        "DPO_EVAL_API_KEY",
        "DPO_EVAL_MODEL",
    ):
        value = userdata.get(key)
        if value:
            os.environ[key] = value
else:
    DATA_ROOT = Path("data")
    OUTPUT_ROOT = Path("data/rl/dpo_notebook")
    WORKDIR = Path(".dpo_job")

WORKDIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(WORKDIR)
os.environ["PYTHONPATH"] = str(WORKDIR / "src")
print("Runtime:", RUNTIME, "| ARMS:", ARMS, "| SEED:", SEED, "| cwd:", WORKDIR)


## Materialize the exact repository contract

In [ ]:
import json

SOURCE_FILES = json.loads('{"benchmarks/compare_dpo_rewriters.py": "\\"\\"\\"Generate and blindly compare DPO-family query rewriters.\\n\\nThis benchmark evaluates the rewriter alone. It never loads a retriever, corpus, index, or\\nanswer generator. Model outputs are cached before the independent Gemini-family judge runs.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport csv\\nimport hashlib\\nimport itertools\\nimport json\\nimport logging\\nimport math\\nimport os\\nimport random\\nimport re\\nimport sys\\nimport time\\nfrom collections import Counter, defaultdict\\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\\nfrom dataclasses import dataclass\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport yaml\\n\\nPROJECT_ROOT = Path(__file__).resolve().parents[1]\\nSRC_DIR = PROJECT_ROOT / \\"src\\"\\nif str(SRC_DIR) not in sys.path:\\n    sys.path.insert(0, str(SRC_DIR))\\n\\nfrom data.questions import load_question, render_question_value  # noqa: E402\\nfrom personalization.profiles import render_profile  # noqa: E402\\nfrom rag.llm import GeminiClient, OpenAICompatClient  # noqa: E402\\nfrom rag.rewriter import DPORewriter, build_rewrite_messages  # noqa: E402\\nfrom rl.dpo_train import load_pairs, validate_splits  # noqa: E402\\n\\nlogger = logging.getLogger(__name__)\\n\\n_JUDGE_SYSTEM = (\\n    \\"You compare two query rewrites for a Persian educational search system. Judge which \\"\\n    \\"rewrite would retrieve the most useful study passages for the stated learner while \\"\\n    \\"preserving the complete question\'s meaning. The gold answer and explanation are context \\"\\n    \\"for relevance only; penalize a rewrite that leaks the answer. Return only strict JSON: \\"\\n    \'{\\"winner\\":\\"A\\",\\"reason\\":\\"short reason\\"}, \'\\n    \'{\\"winner\\":\\"B\\",\\"reason\\":\\"short reason\\"}, or \'\\n    \'{\\"winner\\":\\"TIE\\",\\"reason\\":\\"short reason\\"}.\'\\n)\\n# Structured output pins the reply shape server-side, so `_parse_judgment` never has to\\n# recover a verdict from prose or fenced markdown.\\n_JUDGE_SCHEMA = {\\n    \\"type\\": \\"OBJECT\\",\\n    \\"properties\\": {\\n        \\"winner\\": {\\"type\\": \\"STRING\\", \\"enum\\": [\\"A\\", \\"B\\", \\"TIE\\"]},\\n        \\"reason\\": {\\"type\\": \\"STRING\\"},\\n    },\\n    \\"required\\": [\\"winner\\", \\"reason\\"],\\n    \\"property_ordering\\": [\\"winner\\", \\"reason\\"],\\n}\\n_SAFE_NAME = re.compile(r\\"[A-Za-z0-9_.-]+\\")\\n\\n\\n@dataclass(frozen=True)\\nclass PromptRecord:\\n    question_ref: str\\n    persona_id: str\\n    query: str\\n    profile: str\\n    answer: str\\n    explanation: str\\n\\n    @property\\n    def key(self) -> tuple[str, str]:\\n        return self.question_ref, self.persona_id\\n\\n\\n@dataclass(frozen=True)\\nclass Candidate:\\n    name: str\\n    kind: str\\n    run_dir: Path | None = None\\n\\n\\n@dataclass(frozen=True)\\nclass RetryPolicy:\\n    max_attempts: int\\n    initial_backoff_seconds: float\\n    backoff_multiplier: float\\n    max_backoff_seconds: float\\n\\n    @classmethod\\n    def from_config(cls, config: dict[str, Any]) -> RetryPolicy:\\n        policy = cls(\\n            max_attempts=int(config[\\"max_attempts\\"]),\\n            initial_backoff_seconds=float(config[\\"initial_backoff_seconds\\"]),\\n            backoff_multiplier=float(config[\\"backoff_multiplier\\"]),\\n            max_backoff_seconds=float(config[\\"max_backoff_seconds\\"]),\\n        )\\n        if policy.max_attempts < 1:\\n            raise ValueError(\\"comparison.retry.max_attempts must be at least 1\\")\\n        if policy.initial_backoff_seconds < 0 or policy.backoff_multiplier < 1:\\n            raise ValueError(\\"comparison retry backoff values are invalid\\")\\n        if policy.max_backoff_seconds < policy.initial_backoff_seconds:\\n            raise ValueError(\\"comparison retry maximum must not be below its initial delay\\")\\n        return policy\\n\\n\\ndef _sha256(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open(\\"rb\\") as handle:\\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef _write_json(path: Path, payload: object) -> None:\\n    path.write_text(\\n        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True) + \\"\\\\n\\",\\n        encoding=\\"utf-8\\",\\n    )\\n\\n\\ndef _write_jsonl(path: Path, records: list[dict[str, Any]]) -> None:\\n    with path.open(\\"w\\", encoding=\\"utf-8\\") as handle:\\n        for record in records:\\n            handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True) + \\"\\\\n\\")\\n\\n\\ndef _read_jsonl(path: Path) -> list[dict[str, Any]]:\\n    records = []\\n    for line_number, line in enumerate(path.read_text(encoding=\\"utf-8\\").splitlines(), 1):\\n        if not line.strip():\\n            continue\\n        try:\\n            record = json.loads(line)\\n        except json.JSONDecodeError as exc:\\n            raise ValueError(f\\"{path} line {line_number} is invalid JSON: {exc}\\") from exc\\n        if not isinstance(record, dict):\\n            raise ValueError(f\\"{path} line {line_number} must contain an object\\")\\n        records.append(record)\\n    return records\\n\\n\\ndef _parse_question_ref(question_ref: str) -> tuple[str, str]:\\n    try:\\n        exam_stem, question_id = question_ref.split(\\":\\", 1)\\n    except ValueError as exc:\\n        raise ValueError(f\\"Invalid question_ref {question_ref!r}\\") from exc\\n    if not exam_stem or not question_id:\\n        raise ValueError(f\\"Invalid question_ref {question_ref!r}\\")\\n    return exam_stem, question_id\\n\\n\\ndef load_validation_prompts(config: dict[str, Any]) -> tuple[list[PromptRecord], dict[str, str]]:\\n    \\"\\"\\"Load 272 unique validation question/persona prompts and their judge-only gold fields.\\"\\"\\"\\n    train_path = Path(config[\\"data\\"][\\"train_path\\"])\\n    val_path = Path(config[\\"data\\"][\\"val_path\\"])\\n    train_pairs = load_pairs(train_path)\\n    val_pairs = load_pairs(val_path)\\n    split_report = validate_splits(train_pairs, val_pairs)\\n    questions_dir = Path(config[\\"data\\"][\\"questions_dir\\"])\\n\\n    grouped: dict[tuple[str, str], dict[str, Any]] = {}\\n    for pair in val_pairs:\\n        key = pair[\\"question_ref\\"], pair[\\"persona_id\\"]\\n        previous = grouped.get(key)\\n        if previous is not None and previous[\\"query\\"] != pair[\\"query\\"]:\\n            raise ValueError(f\\"Validation key {key} has inconsistent query text\\")\\n        grouped[key] = pair\\n\\n    prompts = []\\n    for question_ref, persona_id in sorted(grouped):\\n        pair = grouped[(question_ref, persona_id)]\\n        exam_stem, question_id = _parse_question_ref(question_ref)\\n        question = load_question(exam_stem, question_id, questions_dir)\\n        if question.query != pair[\\"query\\"]:\\n            raise ValueError(\\n                f\\"Pair query for {question_ref} differs from data/questions rendered context\\"\\n            )\\n        answer = \\"\\" if question.answer is None else render_question_value(question.answer)\\n        explanation = question.explanation or \\"\\"\\n        prompts.append(\\n            PromptRecord(\\n                question_ref=question_ref,\\n                persona_id=persona_id,\\n                query=pair[\\"query\\"],\\n                profile=render_profile(persona_id),\\n                answer=answer,\\n                explanation=explanation,\\n            )\\n        )\\n    expected = split_report[\\"validation\\"][\\"question_persona_keys\\"]\\n    if len(prompts) != expected:\\n        raise RuntimeError(f\\"Loaded {len(prompts)} prompts, split report expected {expected}\\")\\n    if len(prompts) != 272:\\n        raise ValueError(f\\"This frozen comparison requires 272 validation prompts, got {len(prompts)}\\")\\n    return prompts, {\\"train\\": _sha256(train_path), \\"validation\\": _sha256(val_path)}\\n\\n\\ndef _parse_assignment(raw: str, option: str) -> tuple[str, str]:\\n    if \\"=\\" not in raw:\\n        raise ValueError(f\\"{option} must use NAME=VALUE, got {raw!r}\\")\\n    name, value = raw.split(\\"=\\", 1)\\n    if not _SAFE_NAME.fullmatch(name) or not value:\\n        raise ValueError(f\\"Invalid {option} assignment: {raw!r}\\")\\n    return name, value\\n\\n\\ndef resolve_candidates(run_args: list[str], include_base: bool, include_grok: bool) -> list[Candidate]:\\n    candidates = []\\n    names = set()\\n    for raw in run_args:\\n        name, value = _parse_assignment(raw, \\"--run\\")\\n        if name in {\\"base_qwen\\", \\"grok\\"}:\\n            raise ValueError(f\\"Candidate name {name!r} is reserved for its fixed baseline\\")\\n        if name in names:\\n            raise ValueError(f\\"Duplicate candidate name: {name}\\")\\n        run_dir = Path(value)\\n        if not (run_dir / \\"dpo_best\\").is_dir():\\n            raise FileNotFoundError(f\\"Candidate {name} has no promoted adapter: {run_dir / \'dpo_best\'}\\")\\n        candidates.append(Candidate(name=name, kind=\\"trained\\", run_dir=run_dir))\\n        names.add(name)\\n    if include_base:\\n        candidates.append(Candidate(name=\\"base_qwen\\", kind=\\"base\\"))\\n        names.add(\\"base_qwen\\")\\n    if include_grok:\\n        candidates.append(Candidate(name=\\"grok\\", kind=\\"grok\\"))\\n        names.add(\\"grok\\")\\n    if len(candidates) < 2:\\n        raise ValueError(\\"Comparison requires at least two candidates\\")\\n    return candidates\\n\\n\\ndef _run_manifest(candidate: Candidate) -> dict[str, Any] | None:\\n    if candidate.run_dir is None:\\n        return None\\n    manifest_path = candidate.run_dir / \\"run_manifest.json\\"\\n    if not manifest_path.is_file():\\n        raise FileNotFoundError(f\\"Missing run manifest for {candidate.name}: {manifest_path}\\")\\n    return json.loads(manifest_path.read_text(encoding=\\"utf-8\\"))\\n\\n\\ndef _candidate_metadata(\\n    candidate: Candidate,\\n    config: dict[str, Any],\\n    data_hashes: dict[str, str],\\n) -> dict[str, Any]:\\n    generation = config[\\"generation\\"]\\n    metadata: dict[str, Any] = {\\n        \\"candidate\\": candidate.name,\\n        \\"kind\\": candidate.kind,\\n        \\"data_sha256\\": data_hashes,\\n        \\"prompt_contract\\": \\"build_rewrite_messages+qwen_chat_template+enable_thinking_false\\",\\n    }\\n    if candidate.kind in {\\"trained\\", \\"base\\"}:\\n        metadata[\\"model\\"] = config[\\"model\\"][\\"name\\"]\\n        metadata[\\"generation\\"] = {\\n            \\"max_new_tokens\\": int(generation[\\"max_new_tokens\\"]),\\n            \\"do_sample\\": False,\\n        }\\n    if candidate.kind == \\"trained\\":\\n        manifest = _run_manifest(candidate)\\n        if manifest is None:\\n            raise RuntimeError(f\\"Missing run manifest for {candidate.name}\\")\\n        if manifest.get(\\"data_sha256\\") != data_hashes:\\n            raise ValueError(f\\"Candidate {candidate.name} was trained on different pair-file hashes\\")\\n        metadata.update(\\n            {\\n                \\"arm\\": manifest.get(\\"arm\\"),\\n                \\"seed\\": manifest.get(\\"seed\\"),\\n                \\"adapter_path\\": str(candidate.run_dir / \\"dpo_best\\"),\\n            }\\n        )\\n    elif candidate.kind == \\"grok\\":\\n        metadata[\\"model\\"] = generation[\\"grok_model\\"]\\n        metadata[\\"generation\\"] = {\\n            \\"temperature\\": float(generation[\\"grok_temperature\\"]),\\n            \\"max_new_tokens\\": int(generation[\\"max_new_tokens\\"]),\\n        }\\n    return metadata\\n\\n\\ndef _cache_paths(output_dir: Path, candidate: Candidate) -> tuple[Path, Path]:\\n    cache_dir = output_dir / \\"outputs\\"\\n    return cache_dir / f\\"{candidate.name}.jsonl\\", cache_dir / f\\"{candidate.name}.meta.json\\"\\n\\n\\ndef validate_output_cache(\\n    output_path: Path,\\n    metadata_path: Path,\\n    prompts: list[PromptRecord],\\n    expected_metadata: dict[str, Any],\\n) -> list[dict[str, Any]]:\\n    if not output_path.is_file() or not metadata_path.is_file():\\n        raise FileNotFoundError(f\\"Missing output cache or metadata: {output_path}\\")\\n    metadata = json.loads(metadata_path.read_text(encoding=\\"utf-8\\"))\\n    for key, expected in expected_metadata.items():\\n        if metadata.get(key) != expected:\\n            raise ValueError(\\n                f\\"Cache metadata mismatch for {output_path.name} field {key!r}: \\"\\n                f\\"expected {expected!r}, got {metadata.get(key)!r}\\"\\n            )\\n    if metadata.get(\\"output_sha256\\") != _sha256(output_path):\\n        raise ValueError(f\\"Output hash mismatch for {output_path}\\")\\n\\n    records = _read_jsonl(output_path)\\n    expected = {prompt.key: prompt for prompt in prompts}\\n    actual: dict[tuple[str, str], dict[str, Any]] = {}\\n    for record in records:\\n        key = record.get(\\"question_ref\\"), record.get(\\"persona_id\\")\\n        if key in actual:\\n            raise ValueError(f\\"Duplicate output key {key} in {output_path}\\")\\n        if key not in expected:\\n            raise ValueError(f\\"Unexpected output key {key} in {output_path}\\")\\n        prompt = expected[key]\\n        if record.get(\\"query\\") != prompt.query or record.get(\\"profile\\") != prompt.profile:\\n            raise ValueError(f\\"Prompt mismatch for {key} in {output_path}\\")\\n        rewrite = record.get(\\"rewrite\\")\\n        if not isinstance(rewrite, str) or not rewrite.strip():\\n            raise ValueError(f\\"Empty rewrite for {key} in {output_path}\\")\\n        actual[key] = record\\n    missing = sorted(set(expected) - set(actual))\\n    if missing:\\n        raise ValueError(f\\"Output cache {output_path} is missing {len(missing)} prompts\\")\\n    return [actual[prompt.key] for prompt in prompts]\\n\\n\\ndef _local_rewrites(\\n    candidate: Candidate,\\n    config: dict[str, Any],\\n    prompts: list[PromptRecord],\\n) -> list[str]:\\n    generation_config = config[\\"generation\\"]\\n    rewriter = DPORewriter(\\n        model_name=config[\\"model\\"][\\"name\\"],\\n        adapter_path=str(candidate.run_dir / \\"dpo_best\\") if candidate.run_dir else None,\\n        device=\\"cuda\\",\\n        max_seq_length=int(config[\\"model\\"][\\"max_seq_length\\"]),\\n        generation={\\n            \\"max_new_tokens\\": int(generation_config[\\"max_new_tokens\\"]),\\n            \\"do_sample\\": False,\\n        },\\n    )\\n    batch_size = int(generation_config[\\"batch_size\\"])\\n    rewrites = []\\n    for start in range(0, len(prompts), batch_size):\\n        batch = prompts[start : start + batch_size]\\n        rewrites.extend(\\n            rewriter.rewrite_batch(\\n                [prompt.profile for prompt in batch],\\n                [prompt.query for prompt in batch],\\n            )\\n        )\\n    del rewriter\\n    try:\\n        import torch\\n\\n        torch.cuda.empty_cache()\\n    except (ImportError, RuntimeError):\\n        pass\\n    return rewrites\\n\\n\\ndef _grok_rewrites(config: dict[str, Any], prompts: list[PromptRecord]) -> list[str]:\\n    from data.settings import REWRITER_API_KEY, REWRITER_BASE_URL\\n\\n    if not REWRITER_BASE_URL or not REWRITER_API_KEY:\\n        raise RuntimeError(\\"Grok generation requires REWRITER_BASE_URL and REWRITER_API_KEY\\")\\n    generation = config[\\"generation\\"]\\n    client = OpenAICompatClient(\\n        base_url=REWRITER_BASE_URL,\\n        api_key=REWRITER_API_KEY,\\n        model=generation[\\"grok_model\\"],\\n        temperature=float(generation[\\"grok_temperature\\"]),\\n        max_tokens=int(generation[\\"max_new_tokens\\"]),\\n    )\\n    rewrites = []\\n    for index, prompt in enumerate(prompts, 1):\\n        rewrite = client.chat(build_rewrite_messages(prompt.profile, prompt.query)).strip()\\n        if not rewrite:\\n            raise RuntimeError(f\\"Grok returned an empty rewrite for {prompt.key}\\")\\n        rewrites.append(rewrite)\\n        logger.info(\\"Grok rewrite %d/%d\\", index, len(prompts))\\n    return rewrites\\n\\n\\ndef generate_or_load_outputs(\\n    candidate: Candidate,\\n    config: dict[str, Any],\\n    prompts: list[PromptRecord],\\n    data_hashes: dict[str, str],\\n    output_dir: Path,\\n    *,\\n    force: bool,\\n) -> list[dict[str, Any]]:\\n    output_path, metadata_path = _cache_paths(output_dir, candidate)\\n    expected_metadata = _candidate_metadata(candidate, config, data_hashes)\\n    if output_path.is_file() and metadata_path.is_file() and not force:\\n        return validate_output_cache(output_path, metadata_path, prompts, expected_metadata)\\n\\n    previous_hash = _sha256(output_path) if output_path.is_file() else None\\n    if candidate.kind in {\\"trained\\", \\"base\\"}:\\n        rewrites = _local_rewrites(candidate, config, prompts)\\n    else:\\n        rewrites = _grok_rewrites(config, prompts)\\n    records = [\\n        {\\n            \\"candidate\\": candidate.name,\\n            \\"question_ref\\": prompt.question_ref,\\n            \\"persona_id\\": prompt.persona_id,\\n            \\"query\\": prompt.query,\\n            \\"profile\\": prompt.profile,\\n            \\"rewrite\\": rewrite,\\n        }\\n        for prompt, rewrite in zip(prompts, rewrites, strict=True)\\n    ]\\n    output_path.parent.mkdir(parents=True, exist_ok=True)\\n    _write_jsonl(output_path, records)\\n    output_hash = _sha256(output_path)\\n    if previous_hash and candidate.kind in {\\"trained\\", \\"base\\"} and previous_hash != output_hash:\\n        raise RuntimeError(\\n            f\\"Deterministic Qwen output changed for {candidate.name}: \\"\\n            f\\"{previous_hash} != {output_hash}\\"\\n        )\\n    _write_json(metadata_path, {**expected_metadata, \\"rows\\": len(records), \\"output_sha256\\": output_hash})\\n    return validate_output_cache(output_path, metadata_path, prompts, expected_metadata)\\n\\n\\ndef _parse_pairs(raw_pairs: list[str], candidate_names: list[str]) -> list[tuple[str, str]]:\\n    if not raw_pairs:\\n        return list(itertools.combinations(candidate_names, 2))\\n    known = set(candidate_names)\\n    pairs = []\\n    for raw in raw_pairs:\\n        if \\":\\" not in raw:\\n            raise ValueError(f\\"--pair must use LEFT:RIGHT, got {raw!r}\\")\\n        left, right = raw.split(\\":\\", 1)\\n        if left == right or left not in known or right not in known:\\n            raise ValueError(f\\"Invalid candidate pair: {raw!r}\\")\\n        pair = tuple(sorted((left, right)))\\n        if pair in pairs:\\n            raise ValueError(f\\"Duplicate candidate pair: {pair}\\")\\n        pairs.append(pair)\\n    return pairs\\n\\n\\ndef _job_key(prompt: PromptRecord, left: str, right: str, seed: int) -> str:\\n    raw = f\\"{prompt.question_ref}\\\\0{prompt.persona_id}\\\\0{left}\\\\0{right}\\\\0{seed}\\"\\n    return hashlib.sha256(raw.encode()).hexdigest()\\n\\n\\ndef build_job_manifest(\\n    prompts: list[PromptRecord],\\n    model_pairs: list[tuple[str, str]],\\n    comparison_seed: int,\\n) -> list[dict[str, Any]]:\\n    \\"\\"\\"Build exact 50/50 hidden orientation per model pair using stable hash order.\\"\\"\\"\\n    jobs = []\\n    half = len(prompts) // 2\\n    for left, right in model_pairs:\\n        ranked = sorted(\\n            prompts,\\n            key=lambda prompt: _job_key(prompt, left, right, comparison_seed),\\n        )\\n        left_as_a = {prompt.key for prompt in ranked[:half]}\\n        for prompt in prompts:\\n            candidate_a, candidate_b = (\\n                (left, right) if prompt.key in left_as_a else (right, left)\\n            )\\n            jobs.append(\\n                {\\n                    \\"job_key\\": _job_key(prompt, left, right, comparison_seed),\\n                    \\"question_ref\\": prompt.question_ref,\\n                    \\"persona_id\\": prompt.persona_id,\\n                    \\"model_left\\": left,\\n                    \\"model_right\\": right,\\n                    \\"candidate_a\\": candidate_a,\\n                    \\"candidate_b\\": candidate_b,\\n                    \\"comparison_seed\\": comparison_seed,\\n                }\\n            )\\n    keys = [job[\\"job_key\\"] for job in jobs]\\n    if len(keys) != len(set(keys)):\\n        raise RuntimeError(\\"Comparison job manifest contains duplicate keys\\")\\n    return jobs\\n\\n\\ndef _validate_orientation(jobs: list[dict[str, Any]]) -> dict[str, dict[str, int]]:\\n    counts: dict[tuple[str, str], Counter[str]] = defaultdict(Counter)\\n    for job in jobs:\\n        pair = job[\\"model_left\\"], job[\\"model_right\\"]\\n        counts[pair][job[\\"candidate_a\\"]] += 1\\n    report = {f\\"{left}:{right}\\": dict(counter) for (left, right), counter in counts.items()}\\n    for pair, counter in counts.items():\\n        values = list(counter.values())\\n        if len(values) != 2 or abs(values[0] - values[1]) > 1:\\n            raise RuntimeError(f\\"Unbalanced A/B orientation for {pair}: {dict(counter)}\\")\\n    return report\\n\\n\\ndef _judge_prompt(prompt: PromptRecord, rewrite_a: str, rewrite_b: str) -> str:\\n    gold_sections = []\\n    if prompt.answer:\\n        gold_sections.append(f\\"Gold answer/reference:\\\\n{prompt.answer}\\")\\n    if prompt.explanation:\\n        gold_sections.append(f\\"Gold explanation/rubric:\\\\n{prompt.explanation}\\")\\n    gold = \\"\\\\n\\\\n\\".join(gold_sections) or \\"No gold explanation supplied.\\"\\n    return (\\n        f\\"Learner profile:\\\\n{prompt.profile}\\\\n\\\\n\\"\\n        f\\"Original complete question:\\\\n{prompt.query}\\\\n\\\\n\\"\\n        f\\"Judge-only gold context:\\\\n{gold}\\\\n\\\\n\\"\\n        f\\"Rewrite A:\\\\n{rewrite_a}\\\\n\\\\n\\"\\n        f\\"Rewrite B:\\\\n{rewrite_b}\\"\\n    )\\n\\n\\ndef _parse_judgment(raw: str) -> tuple[str, str]:\\n    stripped = raw.strip().removeprefix(\\"```json\\").removeprefix(\\"```\\").removesuffix(\\"```\\").strip()\\n    result = json.loads(stripped)\\n    if not isinstance(result, dict) or set(result) != {\\"winner\\", \\"reason\\"}:\\n        raise ValueError(\\"Judge response must contain exactly winner and reason\\")\\n    winner = result[\\"winner\\"]\\n    reason = result[\\"reason\\"]\\n    if winner not in {\\"A\\", \\"B\\", \\"TIE\\"}:\\n        raise ValueError(f\\"Invalid judge winner: {winner!r}\\")\\n    if not isinstance(reason, str) or not reason.strip():\\n        raise ValueError(\\"Judge reason must be a nonempty string\\")\\n    return winner, reason.strip()\\n\\n\\nclass JudgeUnavailable(RuntimeError):\\n    \\"\\"\\"The judge endpoint rejected a request in a way no retry can fix.\\"\\"\\"\\n\\n\\ndef _judge_error_is_transient(exc: Exception) -> bool:\\n    \\"\\"\\"Decide whether *exc* is worth another attempt.\\n\\n    google-genai raises ``ClientError``/``ServerError`` carrying an integer ``code``.\\n    Rate limits, timeouts and 5xx are transient; every other 4xx (wrong route, wrong\\n    model name, bad key, revoked quota) will fail identically on every retry and for\\n    every remaining job, so it must abort the run instead of burning the budget.\\n    \\"\\"\\"\\n    if isinstance(exc, ValueError):\\n        # A malformed or truncated reply. Cheap to ask once more.\\n        return True\\n    code = getattr(exc, \\"code\\", None)\\n    if isinstance(code, int):\\n        return code in {408, 429} or code >= 500\\n    # Connection resets and read timeouts arrive without a status code.\\n    return True\\n\\n\\ndef _judge_one(\\n    job: dict[str, Any],\\n    prompts_by_key: dict[tuple[str, str], PromptRecord],\\n    outputs: dict[str, dict[tuple[str, str], str]],\\n    client: GeminiClient,\\n    retry: RetryPolicy,\\n) -> dict[str, Any]:\\n    key = job[\\"question_ref\\"], job[\\"persona_id\\"]\\n    prompt = _judge_prompt(\\n        prompts_by_key[key],\\n        outputs[job[\\"candidate_a\\"]][key],\\n        outputs[job[\\"candidate_b\\"]][key],\\n    )\\n    delay = retry.initial_backoff_seconds\\n    last_error = \\"\\"\\n    for attempt in range(1, retry.max_attempts + 1):\\n        try:\\n            winner, reason = _parse_judgment(client.generate(prompt))\\n            return {**job, \\"status\\": \\"ok\\", \\"winner\\": winner, \\"reason\\": reason, \\"attempts\\": attempt}\\n        except Exception as exc:\\n            last_error = f\\"{type(exc).__name__}: {exc}\\"\\n            if not _judge_error_is_transient(exc):\\n                raise JudgeUnavailable(\\n                    f\\"Judge endpoint failed permanently on attempt {attempt}: {last_error}\\"\\n                ) from exc\\n            if attempt < retry.max_attempts:\\n                time.sleep(delay)\\n                delay = min(delay * retry.backoff_multiplier, retry.max_backoff_seconds)\\n    return {\\n        **job,\\n        \\"status\\": \\"missing\\",\\n        \\"winner\\": None,\\n        \\"reason\\": None,\\n        \\"attempts\\": retry.max_attempts,\\n        \\"error\\": last_error,\\n    }\\n\\n\\ndef run_judge(\\n    config: dict[str, Any],\\n    jobs: list[dict[str, Any]],\\n    prompts: list[PromptRecord],\\n    cached_records: dict[str, list[dict[str, Any]]],\\n    output_dir: Path,\\n) -> list[dict[str, Any]]:\\n    base_url = os.environ.get(\\"DPO_EVAL_BASE_URL\\")\\n    api_key = os.environ.get(\\"DPO_EVAL_API_KEY\\")\\n    model = os.environ.get(\\"DPO_EVAL_MODEL\\")\\n    missing_env = [\\n        name\\n        for name, value in (\\n            (\\"DPO_EVAL_BASE_URL\\", base_url),\\n            (\\"DPO_EVAL_API_KEY\\", api_key),\\n            (\\"DPO_EVAL_MODEL\\", model),\\n        )\\n        if not value\\n    ]\\n    if missing_env:\\n        raise RuntimeError(\\"Missing required comparison environment: \\" + \\", \\".join(missing_env))\\n    if \\"gemini\\" not in model.lower():\\n        raise ValueError(\\"DPO_EVAL_MODEL must identify the agreed Gemini-family judge\\")\\n\\n    results_path = output_dir / \\"judgments.jsonl\\"\\n    existing = _read_jsonl(results_path) if results_path.is_file() else []\\n    existing_by_key = {record[\\"job_key\\"]: record for record in existing}\\n    expected_keys = {job[\\"job_key\\"] for job in jobs}\\n    unexpected = set(existing_by_key) - expected_keys\\n    if unexpected:\\n        raise ValueError(f\\"Judgment cache contains {len(unexpected)} jobs outside this manifest\\")\\n\\n    prompts_by_key = {prompt.key: prompt for prompt in prompts}\\n    outputs = {\\n        name: {\\n            (record[\\"question_ref\\"], record[\\"persona_id\\"]): record[\\"rewrite\\"]\\n            for record in records\\n        }\\n        for name, records in cached_records.items()\\n    }\\n    retry = RetryPolicy.from_config(config[\\"comparison\\"][\\"retry\\"])\\n    client = GeminiClient(\\n        base_url=base_url,\\n        api_key=api_key,\\n        model=model,\\n        system_instruction=_JUDGE_SYSTEM,\\n        temperature=0.0,\\n        max_output_tokens=int(config[\\"comparison\\"][\\"judge_max_tokens\\"]),\\n        # An absent key means the documented default; an explicit empty value hands the\\n        # choice to the model, which matters because the level a model accepts depends on\\n        # the model, and DPO_EVAL_MODEL is a runtime value.\\n        thinking_level=config[\\"comparison\\"].get(\\"judge_thinking_level\\", \\"minimal\\"),\\n        response_schema=_JUDGE_SCHEMA,\\n    )\\n    pending = [job for job in jobs if job[\\"job_key\\"] not in existing_by_key]\\n    max_workers = int(config[\\"comparison\\"][\\"max_workers\\"])\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    fatal: JudgeUnavailable | None = None\\n    with ThreadPoolExecutor(max_workers=max_workers) as executor:\\n        futures = {\\n            executor.submit(_judge_one, job, prompts_by_key, outputs, client, retry): job\\n            for job in pending\\n        }\\n        for index, future in enumerate(as_completed(futures), 1):\\n            try:\\n                result = future.result()\\n            except JudgeUnavailable as exc:\\n                # Every remaining job would fail the same way. Drop the queue instead of\\n                # spending the budget, and leave no half-empty judgment cache behind.\\n                fatal = exc\\n                executor.shutdown(wait=False, cancel_futures=True)\\n                break\\n            existing_by_key[result[\\"job_key\\"]] = result\\n            with results_path.open(\\"a\\", encoding=\\"utf-8\\") as handle:\\n                handle.write(json.dumps(result, ensure_ascii=False, sort_keys=True) + \\"\\\\n\\")\\n            logger.info(\\"Judgment %d/%d status=%s\\", index, len(pending), result[\\"status\\"])\\n    if fatal is not None:\\n        raise fatal\\n    ordered = [existing_by_key[job[\\"job_key\\"]] for job in jobs]\\n    _write_jsonl(results_path, ordered)\\n    return ordered\\n\\n\\ndef _result_score(result: dict[str, Any], candidate: str) -> float:\\n    if result[\\"winner\\"] == \\"TIE\\":\\n        return 0.5\\n    winner = result[\\"candidate_a\\"] if result[\\"winner\\"] == \\"A\\" else result[\\"candidate_b\\"]\\n    return 1.0 if winner == candidate else 0.0\\n\\n\\ndef _bootstrap_interval(values: list[float], samples: int, seed: int) -> list[float]:\\n    if not values:\\n        return [math.nan, math.nan]\\n    rng = random.Random(seed)\\n    means = []\\n    count = len(values)\\n    for _ in range(samples):\\n        means.append(sum(values[rng.randrange(count)] for _ in range(count)) / count)\\n    means.sort()\\n    return [means[int(0.025 * samples)], means[min(samples - 1, int(0.975 * samples))]]\\n\\n\\ndef _sign_flip_p_value(values: list[float]) -> float:\\n    wins = sum(value > 0.5 for value in values)\\n    losses = sum(value < 0.5 for value in values)\\n    non_ties = wins + losses\\n    if not non_ties:\\n        return 1.0\\n    tail = min(wins, losses)\\n    probability = sum(math.comb(non_ties, k) for k in range(tail + 1)) / (2**non_ties)\\n    return min(1.0, 2.0 * probability)\\n\\n\\ndef _holm_adjust(p_values: dict[str, float]) -> dict[str, float]:\\n    ordered = sorted(p_values, key=p_values.get)\\n    adjusted: dict[str, float] = {}\\n    running = 0.0\\n    total = len(ordered)\\n    for rank, key in enumerate(ordered):\\n        running = max(running, min(1.0, p_values[key] * (total - rank)))\\n        adjusted[key] = running\\n    return adjusted\\n\\n\\ndef summarize_results(\\n    config: dict[str, Any],\\n    candidates: list[Candidate],\\n    results: list[dict[str, Any]],\\n    output_dir: Path,\\n) -> dict[str, Any]:\\n    complete = [result for result in results if result[\\"status\\"] == \\"ok\\"]\\n    missing = len(results) - len(complete)\\n    grouped: dict[tuple[str, str], list[dict[str, Any]]] = defaultdict(list)\\n    for result in complete:\\n        grouped[(result[\\"model_left\\"], result[\\"model_right\\"])].append(result)\\n\\n    bootstrap_samples = int(config[\\"comparison\\"][\\"bootstrap_samples\\"])\\n    comparison_seed = int(results[0][\\"comparison_seed\\"]) if results else 0\\n    head_to_head = []\\n    raw_p_values = {}\\n    candidate_points: dict[str, list[float]] = defaultdict(list)\\n    persona_points: dict[tuple[str, str], list[float]] = defaultdict(list)\\n    for left, right in sorted(grouped):\\n        pair_results = grouped[(left, right)]\\n        left_scores = [_result_score(result, left) for result in pair_results]\\n        counts = Counter(\\"tie\\" if score == 0.5 else \\"win\\" if score == 1.0 else \\"loss\\" for score in left_scores)\\n        pair_key = f\\"{left}:{right}\\"\\n        raw_p_values[pair_key] = _sign_flip_p_value(left_scores)\\n        head_to_head.append(\\n            {\\n                \\"left\\": left,\\n                \\"right\\": right,\\n                \\"n\\": len(left_scores),\\n                \\"missing\\": sum(\\n                    result[\\"status\\"] != \\"ok\\"\\n                    for result in results\\n                    if (result[\\"model_left\\"], result[\\"model_right\\"]) == (left, right)\\n                ),\\n                \\"left_wins\\": counts[\\"win\\"],\\n                \\"ties\\": counts[\\"tie\\"],\\n                \\"left_losses\\": counts[\\"loss\\"],\\n                \\"left_score\\": sum(left_scores) / len(left_scores),\\n                \\"left_score_ci95\\": _bootstrap_interval(\\n                    left_scores,\\n                    bootstrap_samples,\\n                    comparison_seed + int(hashlib.sha256(pair_key.encode()).hexdigest()[:8], 16),\\n                ),\\n                \\"sign_flip_p\\": raw_p_values[pair_key],\\n            }\\n        )\\n        for result, left_score in zip(pair_results, left_scores, strict=True):\\n            candidate_points[left].append(left_score)\\n            candidate_points[right].append(1.0 - left_score)\\n            persona_points[(left, result[\\"persona_id\\"])].append(left_score)\\n            persona_points[(right, result[\\"persona_id\\"])].append(1.0 - left_score)\\n\\n    adjusted = _holm_adjust(raw_p_values)\\n    for row in head_to_head:\\n        row[\\"holm_p\\"] = adjusted[f\\"{row[\'left\']}:{row[\'right\']}\\"]\\n    round_robin = {\\n        candidate.name: (\\n            sum(candidate_points[candidate.name]) / len(candidate_points[candidate.name])\\n            if candidate_points[candidate.name]\\n            else math.nan\\n        )\\n        for candidate in candidates\\n    }\\n    per_persona = [\\n        {\\n            \\"candidate\\": candidate,\\n            \\"persona_id\\": persona,\\n            \\"score\\": sum(values) / len(values),\\n            \\"n\\": len(values),\\n        }\\n        for (candidate, persona), values in sorted(persona_points.items())\\n    ]\\n\\n    selected_variant = None\\n    if \\"wpo\\" in round_robin and \\"robust_dpo\\" in round_robin:\\n        wpo_score = round_robin[\\"wpo\\"]\\n        robust_score = round_robin[\\"robust_dpo\\"]\\n        if wpo_score != robust_score:\\n            selected_variant = \\"wpo\\" if wpo_score > robust_score else \\"robust_dpo\\"\\n        else:\\n            direct = next(\\n                row\\n                for row in head_to_head\\n                if {row[\\"left\\"], row[\\"right\\"]} == {\\"wpo\\", \\"robust_dpo\\"}\\n            )\\n            wpo_direct = direct[\\"left_score\\"] if direct[\\"left\\"] == \\"wpo\\" else 1 - direct[\\"left_score\\"]\\n            selected_variant = \\"wpo\\" if wpo_direct >= 0.5 else \\"robust_dpo\\"\\n\\n    summary = {\\n        \\"jobs\\": len(results),\\n        \\"complete\\": len(complete),\\n        \\"missing\\": missing,\\n        \\"round_robin\\": round_robin,\\n        \\"head_to_head\\": head_to_head,\\n        \\"per_persona\\": per_persona,\\n        \\"selected_variant\\": selected_variant,\\n        \\"selection_rule\\": (\\n            \\"round-robin score, then direct head-to-head, then WPO for the known off-policy gap\\"\\n        ),\\n    }\\n    _write_json(output_dir / \\"summary.json\\", summary)\\n    with (output_dir / \\"head_to_head.csv\\").open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as handle:\\n        fieldnames = [\\n            \\"left\\",\\n            \\"right\\",\\n            \\"n\\",\\n            \\"missing\\",\\n            \\"left_wins\\",\\n            \\"ties\\",\\n            \\"left_losses\\",\\n            \\"left_score\\",\\n            \\"left_score_ci95\\",\\n            \\"sign_flip_p\\",\\n            \\"holm_p\\",\\n        ]\\n        writer = csv.DictWriter(handle, fieldnames=fieldnames)\\n        writer.writeheader()\\n        writer.writerows(head_to_head)\\n    with (output_dir / \\"per_persona.csv\\").open(\\"w\\", encoding=\\"utf-8\\", newline=\\"\\") as handle:\\n        writer = csv.DictWriter(handle, fieldnames=[\\"candidate\\", \\"persona_id\\", \\"score\\", \\"n\\"])\\n        writer.writeheader()\\n        writer.writerows(per_persona)\\n    return summary\\n\\n\\ndef _parse_args() -> argparse.Namespace:\\n    parser = argparse.ArgumentParser(description=\\"Compare promoted DPO-family rewriters\\")\\n    parser.add_argument(\\"--config\\", type=Path, required=True)\\n    parser.add_argument(\\n        \\"--run\\",\\n        action=\\"append\\",\\n        default=[],\\n        metavar=\\"NAME=RUN_DIR\\",\\n        help=\\"Candidate run directory containing dpo_best and run_manifest.json\\",\\n    )\\n    parser.add_argument(\\n        \\"--pair\\",\\n        action=\\"append\\",\\n        default=[],\\n        metavar=\\"LEFT:RIGHT\\",\\n        help=\\"Judge only this model pair; repeat as needed. Defaults to all pairs.\\",\\n    )\\n    parser.add_argument(\\"--comparison-seed\\", type=int)\\n    parser.add_argument(\\"--output-dir\\", type=Path)\\n    parser.add_argument(\\"--without-base\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--without-grok\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--force-regenerate\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--generate-only\\", action=\\"store_true\\")\\n    parser.add_argument(\\"--prepare-only\\", action=\\"store_true\\")\\n    return parser.parse_args()\\n\\n\\ndef _validate_manifest_reuse(\\n    manifest_path: Path,\\n    current: dict[str, Any],\\n    *,\\n    require_judge_identity: bool,\\n) -> dict[str, Any] | None:\\n    if not manifest_path.is_file():\\n        return None\\n    existing = json.loads(manifest_path.read_text(encoding=\\"utf-8\\"))\\n    for key in (\\n        \\"comparison_seed\\",\\n        \\"data_sha256\\",\\n        \\"candidates\\",\\n        \\"model_pairs\\",\\n        \\"prompts\\",\\n        \\"pairs\\",\\n        \\"jobs\\",\\n        \\"orientation\\",\\n        \\"output_hashes\\",\\n    ):\\n        if existing.get(key) != current.get(key):\\n            raise ValueError(f\\"Existing judge manifest has mismatched {key!r}\\")\\n    if require_judge_identity:\\n        for key in (\\"judge_base_url\\", \\"judge_model\\"):\\n            previous = existing.get(key)\\n            if previous is not None and previous != current.get(key):\\n                raise ValueError(f\\"Existing judgments use a different {key!r}: {previous!r}\\")\\n    return existing\\n\\n\\ndef main() -> None:\\n    logging.basicConfig(level=logging.INFO, format=\\"%(asctime)s [%(levelname)s] %(message)s\\")\\n    args = _parse_args()\\n    if args.generate_only and args.prepare_only:\\n        raise ValueError(\\"--generate-only and --prepare-only are mutually exclusive\\")\\n    config = yaml.safe_load(args.config.read_text(encoding=\\"utf-8\\"))\\n    comparison_seed = (\\n        int(config[\\"comparison\\"][\\"screening_seed\\"])\\n        if args.comparison_seed is None\\n        else args.comparison_seed\\n    )\\n    output_dir = args.output_dir or (\\n        Path(config[\\"comparison\\"][\\"output_root\\"]) / f\\"seed-{comparison_seed}\\"\\n    )\\n    output_dir.mkdir(parents=True, exist_ok=True)\\n    prompts, data_hashes = load_validation_prompts(config)\\n    candidates = resolve_candidates(args.run, not args.without_base, not args.without_grok)\\n\\n    cached_records = {}\\n    for candidate in candidates:\\n        if args.prepare_only:\\n            output_path, metadata_path = _cache_paths(output_dir, candidate)\\n            cached_records[candidate.name] = validate_output_cache(\\n                output_path,\\n                metadata_path,\\n                prompts,\\n                _candidate_metadata(candidate, config, data_hashes),\\n            )\\n        else:\\n            cached_records[candidate.name] = generate_or_load_outputs(\\n                candidate,\\n                config,\\n                prompts,\\n                data_hashes,\\n                output_dir,\\n                force=args.force_regenerate,\\n            )\\n    if args.generate_only:\\n        logger.info(\\"Generated or validated %d candidate caches\\", len(candidates))\\n        return\\n\\n    model_pairs = _parse_pairs(args.pair, [candidate.name for candidate in candidates])\\n    jobs = build_job_manifest(prompts, model_pairs, comparison_seed)\\n    orientation = _validate_orientation(jobs)\\n    _write_jsonl(output_dir / \\"judge_jobs.jsonl\\", jobs)\\n    judge_manifest = {\\n        \\"comparison_seed\\": comparison_seed,\\n        \\"data_sha256\\": data_hashes,\\n        \\"candidates\\": [candidate.name for candidate in candidates],\\n        \\"model_pairs\\": [list(pair) for pair in model_pairs],\\n        \\"prompts\\": len(prompts),\\n        \\"pairs\\": len(model_pairs),\\n        \\"jobs\\": len(jobs),\\n        \\"orientation\\": orientation,\\n        \\"judge_base_url\\": os.environ.get(\\"DPO_EVAL_BASE_URL\\"),\\n        \\"judge_model\\": os.environ.get(\\"DPO_EVAL_MODEL\\"),\\n        \\"judge_api_key_present\\": bool(os.environ.get(\\"DPO_EVAL_API_KEY\\")),\\n        \\"output_hashes\\": {\\n            candidate.name: json.loads(_cache_paths(output_dir, candidate)[1].read_text())[\\"output_sha256\\"]\\n            for candidate in candidates\\n        },\\n    }\\n    manifest_path = output_dir / \\"judge_manifest.json\\"\\n    if not args.prepare_only:\\n        missing_judge_env = [\\n            key\\n            for key in (\\"DPO_EVAL_BASE_URL\\", \\"DPO_EVAL_API_KEY\\", \\"DPO_EVAL_MODEL\\")\\n            if not os.environ.get(key)\\n        ]\\n        if missing_judge_env:\\n            raise RuntimeError(\\n                \\"Missing required comparison environment: \\" + \\", \\".join(missing_judge_env)\\n            )\\n    existing_manifest = _validate_manifest_reuse(\\n        manifest_path,\\n        judge_manifest,\\n        require_judge_identity=not args.prepare_only,\\n    )\\n    if args.prepare_only and existing_manifest is not None:\\n        for key in (\\"judge_base_url\\", \\"judge_model\\", \\"judge_api_key_present\\"):\\n            judge_manifest[key] = existing_manifest.get(key)\\n    _write_json(manifest_path, judge_manifest)\\n    print(\\n        f\\"Prepared {len(prompts)} prompts, {len(model_pairs)} model pairs, \\"\\n        f\\"{len(jobs)} judge jobs in {output_dir}\\"\\n    )\\n    if args.prepare_only:\\n        return\\n\\n    results = run_judge(config, jobs, prompts, cached_records, output_dir)\\n    summary = summarize_results(config, candidates, results, output_dir)\\n    print(json.dumps(summary, ensure_ascii=False, indent=2, sort_keys=True))\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "src/data/questions.py": "\\"\\"\\"Render a question\'s complete retrieval context — one definition, shared by every stage.\\n\\nBoth data-generation stages and the models they feed must agree on the exact string that\\nrepresents a question. The retriever LoRA was distilled on the form produced here\\n(``Passage:`` / ``Question:`` / ``Options:`` / ``Pairs:`` / ``Items:`` sections), so a\\ncaller that reconstructs a query from ``stem`` alone silently queries the encoder with a\\nform it never saw in training.\\n\\nGold fields (``answer``, ``explanation``) travel beside the query in ``QuestionContext``\\nand are deliberately *not* part of it: they are judge-only reference material, and\\nfolding them into a retrieval query would leak the answer into the retriever\'s input.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom dataclasses import dataclass\\nfrom typing import TYPE_CHECKING\\n\\nif TYPE_CHECKING:\\n    from pathlib import Path\\n\\n\\n@dataclass(frozen=True)\\nclass QuestionContext:\\n    \\"\\"\\"A question\'s retrieval query plus its judge-only gold fields.\\"\\"\\"\\n\\n    query: str\\n    answer: object | None = None\\n    explanation: str | None = None\\n\\n\\ndef render_question_value(value: object) -> str:\\n    \\"\\"\\"Render an extracted value without changing string content.\\"\\"\\"\\n    if isinstance(value, str):\\n        return value\\n    return json.dumps(value, ensure_ascii=False, sort_keys=True)\\n\\n\\ndef render_numbered_section(label: str, values: object) -> str | None:\\n    if values is None:\\n        return None\\n    entries = values if isinstance(values, list) else [values]\\n    if not entries:\\n        return None\\n    rendered = \\"\\\\n\\".join(\\n        f\\"{index}. {render_question_value(value)}\\" for index, value in enumerate(entries, start=1)\\n    )\\n    return f\\"{label}:\\\\n{rendered}\\"\\n\\n\\ndef load_question(exam_stem: str, qid: str, questions_dir: Path) -> QuestionContext:\\n    \\"\\"\\"Load and render the complete retrieval context for *qid*.\\"\\"\\"\\n    qfile = questions_dir / f\\"{exam_stem}.json\\"\\n    if not qfile.exists():\\n        raise FileNotFoundError(f\\"Question file not found: {qfile}\\")\\n    data = json.loads(qfile.read_text(encoding=\\"utf-8\\"))\\n    for q in data.get(\\"questions\\", []):\\n        if q[\\"id\\"] == qid:\\n            sections: list[str] = []\\n            group_id = q.get(\\"group_id\\")\\n            if group_id is not None:\\n                passages = data.get(\\"passages\\", data.get(\\"passage\\", []))\\n                matching_passage: object | None = None\\n\\n                if isinstance(passages, dict):\\n                    if passages.get(\\"id\\") == group_id or passages.get(\\"group_id\\") == group_id:\\n                        matching_passage = passages\\n                    else:\\n                        matching_passage = passages.get(group_id)\\n                        if matching_passage is None:\\n                            matching_passage = passages.get(str(group_id))\\n                elif isinstance(passages, list):\\n                    for passage in passages:\\n                        if not isinstance(passage, dict):\\n                            continue\\n                        if passage.get(\\"id\\") == group_id or passage.get(\\"group_id\\") == group_id:\\n                            matching_passage = passage\\n                            break\\n\\n                if matching_passage is None:\\n                    raise KeyError(\\n                        f\\"Question {qid!r} in {qfile} references group_id={group_id!r}, \\"\\n                        \\"but no matching top-level passage exists\\"\\n                    )\\n                if isinstance(matching_passage, dict):\\n                    passage_text = matching_passage.get(\\"text\\", matching_passage.get(\\"passage\\"))\\n                else:\\n                    passage_text = matching_passage\\n                if passage_text is None:\\n                    raise KeyError(\\n                        f\\"Question {qid!r} in {qfile} references group_id={group_id!r}, \\"\\n                        \\"but the matching top-level passage has no text\\"\\n                    )\\n                sections.append(f\\"Passage:\\\\n{render_question_value(passage_text)}\\")\\n\\n            sections.append(f\\"Question:\\\\n{render_question_value(q[\'stem\'])}\\")\\n\\n            options_section = render_numbered_section(\\"Options\\", q.get(\\"options\\"))\\n            if options_section is not None:\\n                sections.append(options_section)\\n\\n            pairs = q.get(\\"pairs\\")\\n            if isinstance(pairs, dict):\\n                for side in (\\"left\\", \\"right\\"):\\n                    pair_section = render_numbered_section(f\\"Pairs ({side})\\", pairs.get(side))\\n                    if pair_section is not None:\\n                        sections.append(pair_section)\\n            elif pairs is not None:\\n                pair_section = render_numbered_section(\\"Pairs\\", pairs)\\n                if pair_section is not None:\\n                    sections.append(pair_section)\\n\\n            items_section = render_numbered_section(\\"Items\\", q.get(\\"items\\"))\\n            if items_section is not None:\\n                sections.append(items_section)\\n\\n            return QuestionContext(\\n                query=\\"\\\\n\\\\n\\".join(sections),\\n                answer=q.get(\\"answer\\"),\\n                explanation=q.get(\\"explanation\\"),\\n            )\\n    raise KeyError(f\\"Question {qid!r} not found in {qfile}\\")\\n", "src/data/settings.py": "\\"\\"\\"Centralized environment access — import settings from here, don\'t read os.environ elsewhere.\\n\\nSecrets and the deployment endpoint come from the environment (a project-root ``.env``,\\nloaded once on import). Experiment parameters stay in the YAML configs, not here.\\n\\"\\"\\"\\n\\nimport os\\n\\nfrom dotenv import load_dotenv\\n\\n# Load the project\'s .env once. find_dotenv walks up from this file, so it works regardless\\n# of the current working directory (e.g. a notebook running from notebooks/). Real environment\\n# variables already set take precedence — load_dotenv does not override them.\\nload_dotenv()\\n\\n# OpenAI-compatible client credentials. \\"local\\" lets keyless local servers work.\\nOPENAI_API_KEY: str = os.environ.get(\\"OPENAI_API_KEY\\", \\"local\\")\\n# Endpoint; None lets the OpenAI SDK fall back to its default (the official API).\\nOPENAI_BASE_URL: str | None = os.environ.get(\\"OPENAI_BASE_URL\\")\\n\\n# Gemini judge credentials.\\nGEMINI_API_KEY: str = os.environ.get(\\"GEMINI_API_KEY\\", \\"\\")\\nGEMINI_ENDPOINT: str | None = os.environ.get(\\"GEMINI_ENDPOINT\\")\\n\\n# Rewriter credentials (separate from judge; e.g. xAI/Grok endpoint).\\nREWRITER_API_KEY: str = os.environ.get(\\"REWRITER_API_KEY\\", \\"\\")\\nREWRITER_BASE_URL: str | None = os.environ.get(\\"REWRITER_BASE_URL\\")\\n", "src/personalization/profiles.py": "\\"\\"\\"Learner persona definitions for the Simurgh RAG pipeline.\\n\\nSchema and rendered texts sourced from docs/personas.md.\\nThree personas (crammer, scholar, steady) are used in train/val;\\nnewcomer is held out for test only.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\n\\n\\n@dataclass(frozen=True)\\nclass Profile:\\n    id: str\\n    comprehension: str  # L1–L4\\n    prior_knowledge: str  # L1–L4\\n    learning_goal: str  # L1–L4\\n    explanation_style: str  # L1–L4\\n    split: str  # \\"train\\" or \\"test\\"\\n    rendered: str  # natural-language prose injected into prompts\\n\\n\\nPERSONAS: dict[str, Profile] = {\\n    \\"crammer\\": Profile(\\n        id=\\"crammer\\",\\n        comprehension=\\"L1\\",\\n        prior_knowledge=\\"L1\\",\\n        learning_goal=\\"L1\\",\\n        explanation_style=\\"L4\\",\\n        split=\\"train\\",\\n        rendered=(\\n            \\"A ninth-grader who finds the textbook hard to follow and has little background \\"\\n            \\"on this topic. Mainly wants to pass the exam — give the answer and what is needed \\"\\n            \\"to score — but it must be spelled out simply, step by step, with examples.\\"\\n        ),\\n    ),\\n    \\"scholar\\": Profile(\\n        id=\\"scholar\\",\\n        comprehension=\\"L4\\",\\n        prior_knowledge=\\"L3\\",\\n        learning_goal=\\"L4\\",\\n        explanation_style=\\"L1\\",\\n        split=\\"train\\",\\n        rendered=(\\n            \\"A ninth-grader who reads dense material easily and has solid background on this \\"\\n            \\"topic. Wants to understand the underlying why and how, and the connections between \\"\\n            \\"ideas. Prefers a terse, high-level treatment without hand-holding or padding.\\"\\n        ),\\n    ),\\n    \\"steady\\": Profile(\\n        id=\\"steady\\",\\n        comprehension=\\"L3\\",\\n        prior_knowledge=\\"L2\\",\\n        learning_goal=\\"L2\\",\\n        explanation_style=\\"L2\\",\\n        split=\\"train\\",\\n        rendered=(\\n            \\"A capable ninth-grader with average background on this topic. Wants a correct \\"\\n            \\"answer with a brief justification, balanced toward exam needs. Does not need \\"\\n            \\"elaborate scaffolding, but does appreciate a one-line reason.\\"\\n        ),\\n    ),\\n    \\"newcomer\\": Profile(\\n        id=\\"newcomer\\",\\n        comprehension=\\"L3\\",\\n        prior_knowledge=\\"L1\\",\\n        learning_goal=\\"L4\\",\\n        explanation_style=\\"L4\\",\\n        split=\\"test\\",\\n        rendered=(\\n            \\"A bright ninth-grader who reads well but is new to this topic. Wants real \\"\\n            \\"understanding — the why and how, not just the answer — and needs worked examples \\"\\n            \\"to bridge the missing background.\\"\\n        ),\\n    ),\\n}\\n\\n\\ndef render_profile(persona_id: str) -> str:\\n    \\"\\"\\"Return the natural-language rendering for *persona_id*.\\"\\"\\"\\n    if persona_id not in PERSONAS:\\n        raise ValueError(f\\"Unknown persona: {persona_id!r}. Valid: {list(PERSONAS)}\\")\\n    return PERSONAS[persona_id].rendered\\n\\n\\ndef train_personas() -> list[Profile]:\\n    \\"\\"\\"Return the three training personas (excludes the test holdout).\\"\\"\\"\\n    return [p for p in PERSONAS.values() if p.split == \\"train\\"]\\n", "src/rag/llm.py": "\\"\\"\\"Thin wrappers over the chat endpoints this project talks to.\\"\\"\\"\\n\\nimport openai\\n\\n\\nclass OpenAICompatClient:\\n    \\"\\"\\"Chat client for OpenAI-compatible routes: OpenAI, LMStudio, Ollama, vLLM, Metis.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        base_url: str | None,\\n        api_key: str,\\n        model: str,\\n        temperature: float = 0.2,\\n        max_tokens: int = 800,\\n        reasoning_effort: str | None = None,\\n    ) -> None:\\n        self.client = openai.OpenAI(base_url=base_url, api_key=api_key)\\n        self.model = model\\n        self.temperature = temperature\\n        self.max_tokens = max_tokens\\n        self.reasoning_effort = reasoning_effort\\n\\n    def chat(self, messages: list[dict[str, str]]) -> str:\\n        \\"\\"\\"Send *messages* and return the assistant reply text.\\"\\"\\"\\n        request = {\\n            \\"model\\": self.model,\\n            \\"messages\\": messages,\\n            \\"temperature\\": self.temperature,\\n            \\"max_completion_tokens\\": self.max_tokens,\\n        }\\n        if self.reasoning_effort is not None:\\n            request[\\"reasoning_effort\\"] = self.reasoning_effort\\n\\n        response = self.client.chat.completions.create(**request)\\n        return response.choices[0].message.content or \\"\\"\\n\\n\\nclass GeminiClient:\\n    \\"\\"\\"Gemini chat client for Metis\'s native Google GenAI route.\\n\\n    Metis serves the Gemini family through Google\'s own protocol\\n    (``POST {base_url}/v1beta/models/{model}:generateContent``), not through its\\n    OpenAI-compatible route, so these models are unreachable with\\n    :class:`OpenAICompatClient`.\\n\\n    Automatic function calling is disabled: this client passes no tools, and leaving it on\\n    wraps every request in a tool-calling loop that logs a line per call.\\n    \\"\\"\\"\\n\\n    #: Levels the Gemini API accepts. Which subset a given model supports is model\\n    #: dependent, so the caller picks the value and the server has final say. Gemini 3.5\\n    #: and newer reject the older ``thinking_budget`` field outright.\\n    THINKING_LEVELS = (\\"minimal\\", \\"low\\", \\"medium\\", \\"high\\")\\n\\n    def __init__(\\n        self,\\n        base_url: str | None,\\n        api_key: str,\\n        model: str,\\n        system_instruction: str | None = None,\\n        temperature: float = 0.0,\\n        max_output_tokens: int = 256,\\n        thinking_level: str | None = \\"minimal\\",\\n        response_schema: dict | None = None,\\n    ) -> None:\\n        try:\\n            from google import genai\\n            from google.genai.types import (\\n                AutomaticFunctionCallingConfig,\\n                GenerateContentConfig,\\n                HttpOptions,\\n                ThinkingConfig,\\n            )\\n        except ImportError as exc:  # pragma: no cover - dependency guard\\n            raise RuntimeError(\\n                \\"Gemini models need the google-genai SDK. Install it with:\\\\n\\"\\n                \\"  uv pip install google-genai\\"\\n            ) from exc\\n\\n        # The SDK enum is case-insensitive and accepts unknown values, so a typo would\\n        # only surface as a server-side 400 once per call. Reject it here instead.\\n        level = (thinking_level or \\"\\").strip().lower()\\n        if level and level not in self.THINKING_LEVELS:\\n            raise ValueError(\\n                f\\"Unsupported thinking level {thinking_level!r}; \\"\\n                f\\"use one of {\', \'.join(self.THINKING_LEVELS)}, or an empty value to let \\"\\n                \\"the model choose\\"\\n            )\\n\\n        http_options = HttpOptions(base_url=base_url) if base_url else None\\n        self.client = genai.Client(api_key=api_key, http_options=http_options)\\n        self.model = model\\n        self.config = GenerateContentConfig(\\n            system_instruction=system_instruction,\\n            temperature=temperature,\\n            max_output_tokens=max_output_tokens,\\n            thinking_config=ThinkingConfig(thinking_level=level.upper()) if level else None,\\n            response_mime_type=\\"application/json\\" if response_schema else None,\\n            response_schema=response_schema,\\n            automatic_function_calling=AutomaticFunctionCallingConfig(disable=True),\\n        )\\n\\n    def generate(self, prompt: str) -> str:\\n        \\"\\"\\"Send *prompt* as a single user turn and return the reply text.\\n\\n        Returns an empty string when the model produced no text, which includes a reply\\n        truncated by ``max_output_tokens``: thinking tokens draw from the same budget, and\\n        no Gemini 3.x model lets thinking be turned off entirely.\\n        \\"\\"\\"\\n        response = self.client.models.generate_content(\\n            model=self.model,\\n            contents=prompt,\\n            config=self.config,\\n        )\\n        return response.text or \\"\\"\\n", "src/rag/rewriter.py": "\\"\\"\\"Persona-conditioned query rewriters.\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import TYPE_CHECKING, Any\\n\\nif TYPE_CHECKING:\\n    from rag.llm import OpenAICompatClient\\n\\n_REWRITE_SYSTEM = (\\n    \\"You are a query rewriting assistant for a Persian educational RAG system. \\"\\n    \\"Given a learner profile and an original question, rewrite the question as a \\"\\n    \\"retrieval query that will surface the most pedagogically useful passages for \\"\\n    \\"that specific learner. \\"\\n    \\"Rules: output ONLY the rewritten query — no explanation, no preamble, no quotes. \\"\\n    \\"Keep it in Persian if the original is Persian. \\"\\n    \\"You may expand abbreviations, add prerequisite terms, or rephrase for clarity, \\"\\n    \\"but do not invent facts or change the question\'s intent.\\"\\n)\\n_DEFAULT_GENERATION = {\\"max_new_tokens\\": 224, \\"do_sample\\": False}\\n_ALLOWED_GENERATION_KEYS = {\\n    \\"max_new_tokens\\",\\n    \\"do_sample\\",\\n    \\"temperature\\",\\n    \\"top_p\\",\\n    \\"top_k\\",\\n    \\"repetition_penalty\\",\\n}\\n\\n\\ndef build_rewrite_messages(profile_rendered: str, query: str) -> list[dict[str, str]]:\\n    \\"\\"\\"Build the shared training, local-inference, and remote-baseline prompt.\\"\\"\\"\\n    if not profile_rendered.strip():\\n        raise ValueError(\\"profile_rendered must be nonempty\\")\\n    if not query.strip():\\n        raise ValueError(\\"query must be nonempty\\")\\n    user = (\\n        f\\"Learner profile: {profile_rendered}\\\\n\\\\n\\"\\n        f\\"Original question: {query}\\\\n\\\\n\\"\\n        \\"Rewritten retrieval query:\\"\\n    )\\n    return [\\n        {\\"role\\": \\"system\\", \\"content\\": _REWRITE_SYSTEM},\\n        {\\"role\\": \\"user\\", \\"content\\": user},\\n    ]\\n\\n\\ndef _normalize_generation(generation: dict[str, Any] | None) -> dict[str, Any]:\\n    settings = {**_DEFAULT_GENERATION, **(generation or {})}\\n    unknown = sorted(set(settings) - _ALLOWED_GENERATION_KEYS)\\n    if unknown:\\n        raise ValueError(f\\"Unknown generation settings: {unknown}\\")\\n    max_new_tokens = settings.get(\\"max_new_tokens\\")\\n    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int) or max_new_tokens < 1:\\n        raise ValueError(\\"generation.max_new_tokens must be a positive integer\\")\\n    do_sample = settings.get(\\"do_sample\\")\\n    if not isinstance(do_sample, bool):\\n        raise ValueError(\\"generation.do_sample must be boolean\\")\\n    if not do_sample:\\n        settings.pop(\\"temperature\\", None)\\n        settings.pop(\\"top_p\\", None)\\n        settings.pop(\\"top_k\\", None)\\n    return settings\\n\\n\\ndef generate_rewrite_batch(\\n    model: Any,\\n    tokenizer: Any,\\n    profiles: list[str],\\n    queries: list[str],\\n    *,\\n    generation: dict[str, Any] | None = None,\\n) -> list[str]:\\n    \\"\\"\\"Generate response-only rewrites for a batch of profile/query pairs.\\"\\"\\"\\n    if not profiles or len(profiles) != len(queries):\\n        raise ValueError(\\"profiles and queries must be nonempty lists of equal length\\")\\n    settings = _normalize_generation(generation)\\n    prompt_texts = [\\n        tokenizer.apply_chat_template(\\n            build_rewrite_messages(profile, query),\\n            tokenize=False,\\n            add_generation_prompt=True,\\n            enable_thinking=False,\\n        )\\n        for profile, query in zip(profiles, queries, strict=True)\\n    ]\\n    previous_padding_side = tokenizer.padding_side\\n    tokenizer.padding_side = \\"left\\"\\n    try:\\n        inputs = tokenizer(prompt_texts, return_tensors=\\"pt\\", padding=True)\\n    finally:\\n        tokenizer.padding_side = previous_padding_side\\n    inputs = inputs.to(model.device)\\n    padded_input_length = inputs[\\"input_ids\\"].shape[1]\\n    outputs = model.generate(\\n        **inputs,\\n        **settings,\\n        pad_token_id=tokenizer.pad_token_id,\\n        eos_token_id=tokenizer.eos_token_id,\\n    )\\n    completion_ids = outputs[:, padded_input_length:]\\n    rewrites = tokenizer.batch_decode(completion_ids, skip_special_tokens=True)\\n    cleaned = [rewrite.strip() for rewrite in rewrites]\\n    empty_indices = [index for index, rewrite in enumerate(cleaned) if not rewrite]\\n    if empty_indices:\\n        raise RuntimeError(f\\"Model returned empty rewrites at batch indices {empty_indices}\\")\\n    return cleaned\\n\\n\\nclass PromptedRewriter:\\n    \\"\\"\\"Call a remote LLM for a persona-conditioned query rewrite.\\"\\"\\"\\n\\n    def __init__(self, llm: OpenAICompatClient) -> None:\\n        self.llm = llm\\n\\n    def rewrite(self, profile_rendered: str, query: str) -> str:\\n        messages = build_rewrite_messages(profile_rendered, query)\\n        rewrite = self.llm.chat(messages).strip()\\n        if not rewrite:\\n            raise RuntimeError(\\"Remote rewriter returned an empty completion\\")\\n        return rewrite\\n\\n\\nclass DPORewriter:\\n    \\"\\"\\"Run a Qwen3 base model or a promoted DPO-family LoRA adapter.\\"\\"\\"\\n\\n    def __init__(\\n        self,\\n        model_name: str = \\"Qwen/Qwen3-4B\\",\\n        adapter_path: str | None = None,\\n        device: str = \\"cuda\\",\\n        max_seq_length: int = 768,\\n        generation: dict[str, Any] | None = None,\\n    ) -> None:\\n        import torch\\n        from unsloth import FastLanguageModel\\n\\n        requested_device = torch.device(device)\\n        loader_kwargs: dict[str, Any] = {}\\n        if requested_device.type == \\"cuda\\":\\n            if not torch.cuda.is_available():\\n                raise RuntimeError(f\\"Requested {device}, but CUDA is unavailable\\")\\n            device_index = requested_device.index if requested_device.index is not None else 0\\n            torch.cuda.set_device(device_index)\\n            resolved_device = torch.device(\\"cuda\\", device_index)\\n            load_in_4bit = True\\n            # One whole replica on one GPU. Left to itself, Unsloth\'s planner spreads the\\n            # model over every visible GPU and pins the tied embedding to the output head\'s\\n            # device, so generation feeds token ids on one GPU into an embedding on another.\\n            loader_kwargs[\\"device_map\\"] = {\\"\\": device_index}\\n        elif requested_device.type == \\"cpu\\":\\n            resolved_device = torch.device(\\"cpu\\")\\n            load_in_4bit = False\\n        else:\\n            raise ValueError(f\\"Unsupported DPO rewriter device: {device!r}\\")\\n\\n        normalized_generation = _normalize_generation(generation)\\n        model, tokenizer = FastLanguageModel.from_pretrained(\\n            model_name=model_name,\\n            load_in_4bit=load_in_4bit,\\n            max_seq_length=max_seq_length,\\n            **loader_kwargs,\\n        )\\n        if tokenizer.pad_token is None:\\n            tokenizer.pad_token = tokenizer.eos_token\\n        if adapter_path is not None:\\n            from peft import PeftModel\\n\\n            model = PeftModel.from_pretrained(model, adapter_path)\\n        if resolved_device.type == \\"cpu\\":\\n            model = model.to(resolved_device)\\n        FastLanguageModel.for_inference(model)\\n\\n        placements = {parameter.device for parameter in model.parameters()}\\n        if placements != {resolved_device}:\\n            raise RuntimeError(\\n                f\\"DPO rewriter must hold one replica on {resolved_device}, but its weights \\"\\n                \\"are spread over \\" + \\", \\".join(sorted(str(place) for place in placements))\\n            )\\n        self.model_name = model_name\\n        self.adapter_path = adapter_path\\n        self.max_seq_length = max_seq_length\\n        self.generation = normalized_generation\\n        self.model = model\\n        self.tokenizer = tokenizer\\n\\n    def rewrite_batch(self, profiles: list[str], queries: list[str]) -> list[str]:\\n        return generate_rewrite_batch(\\n            self.model,\\n            self.tokenizer,\\n            profiles,\\n            queries,\\n            generation=self.generation,\\n        )\\n\\n    def rewrite(self, profile_rendered: str, query: str) -> str:\\n        return self.rewrite_batch([profile_rendered], [query])[0]\\n", "src/rl/dpo_train.py": "\\"\\"\\"Train the persona-conditioned Qwen3 query rewriter with DPO-family objectives.\\n\\nThe module owns the data, token-budget, objective, and distributed-training contract used\\nby both the command-line entry point and the generated Kaggle notebook.\\n\\"\\"\\"\\n\\nfrom __future__ import annotations\\n\\nimport argparse\\nimport hashlib\\nimport importlib.metadata\\nimport inspect\\nimport json\\nimport logging\\nimport math\\nimport os\\nimport random\\nimport statistics\\nfrom collections import Counter\\nfrom pathlib import Path\\nfrom typing import TYPE_CHECKING, Any\\n\\nimport yaml\\n\\nif TYPE_CHECKING:\\n    from transformers import PreTrainedTokenizerBase\\n\\nlogger = logging.getLogger(__name__)\\n\\n_REQUIRED_FIELDS = (\\"question_ref\\", \\"persona_id\\", \\"query\\", \\"chosen\\", \\"rejected\\")\\n_PINNED_PACKAGES = {\\n    \\"unsloth\\": \\"2026.8.22\\",\\n    \\"trl\\": \\"0.24.0\\",\\n    \\"transformers\\": \\"4.57.6\\",\\n    \\"datasets\\": \\"4.3.0\\",\\n    \\"peft\\": \\"0.18.0\\",\\n    \\"accelerate\\": \\"1.14.0\\",\\n}\\n_REQUIRED_DPO_PARAMETERS = {\\n    \\"max_prompt_length\\",\\n    \\"max_completion_length\\",\\n    \\"use_weighting\\",\\n    \\"precompute_ref_log_probs\\",\\n    \\"rpo_alpha\\",\\n}\\n# Every field here is part of an arm\'s identity, so `resolve_arm` demands an exact match\\n# against the config. `rpo_alpha` adds TRL\'s supervised NLL term on the chosen response;\\n# without it the objective only widens the chosen/rejected margin, which the seed-42\\n# screening run satisfied by pushing both log-probabilities down (see\\n# docs/results/dpo-arms-seed42-v1.md). TRL scales the anchor by the WPO weight in the `wpo`\\n# arm, so the same alpha is a weaker anchor there; that asymmetry is left explicit.\\n_ARM_CONTRACT = {\\n    \\"dpo\\": {\\n        \\"loss_type\\": \\"sigmoid\\",\\n        \\"use_weighting\\": False,\\n        \\"label_smoothing\\": 0.0,\\n        \\"rpo_alpha\\": 1.0,\\n    },\\n    \\"wpo\\": {\\n        \\"loss_type\\": \\"sigmoid\\",\\n        \\"use_weighting\\": True,\\n        \\"label_smoothing\\": 0.0,\\n        \\"rpo_alpha\\": 1.0,\\n    },\\n    \\"robust_dpo\\": {\\n        \\"loss_type\\": \\"robust\\",\\n        \\"use_weighting\\": False,\\n        \\"label_smoothing\\": 0.1,\\n        \\"rpo_alpha\\": 1.0,\\n    },\\n}\\n_INSTALL_COMMAND = (\\n    \\"pip uninstall -y torchao && \\"\\n    \\"pip install unsloth==2026.8.22 trl==0.24.0 transformers==4.57.6 \\"\\n    \\"datasets==4.3.0 peft==0.18.0 accelerate==1.14.0\\"\\n)\\n\\n\\ndef _is_rank_zero() -> bool:\\n    return int(os.environ.get(\\"RANK\\", \\"0\\")) == 0\\n\\n\\ndef _sha256(path: Path) -> str:\\n    digest = hashlib.sha256()\\n    with path.open(\\"rb\\") as handle:\\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\\"\\"):\\n            digest.update(chunk)\\n    return digest.hexdigest()\\n\\n\\ndef _json_default(value: object) -> object:\\n    if hasattr(value, \\"item\\"):\\n        return value.item()\\n    if isinstance(value, Path):\\n        return str(value)\\n    raise TypeError(f\\"Cannot serialize {type(value).__name__}\\")\\n\\n\\ndef _write_json(path: Path, payload: object) -> None:\\n    path.write_text(\\n        json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True, default=_json_default)\\n        + \\"\\\\n\\",\\n        encoding=\\"utf-8\\",\\n    )\\n\\n\\ndef _verify_training_packages() -> dict[str, str]:\\n    versions: dict[str, str] = {}\\n    mismatches: list[str] = []\\n    for package, expected in _PINNED_PACKAGES.items():\\n        try:\\n            actual = importlib.metadata.version(package)\\n        except importlib.metadata.PackageNotFoundError:\\n            actual = \\"missing\\"\\n        versions[package] = actual\\n        if actual != expected:\\n            mismatches.append(f\\"{package}=={actual} (expected {expected})\\")\\n\\n    try:\\n        torchao_version = importlib.metadata.version(\\"torchao\\")\\n    except importlib.metadata.PackageNotFoundError:\\n        versions[\\"torchao\\"] = \\"not installed\\"\\n    else:\\n        versions[\\"torchao\\"] = torchao_version\\n        mismatches.append(\\n            f\\"torchao=={torchao_version} is installed, but this bitsandbytes QLoRA path \\"\\n            \\"requires the optional TorchAO PEFT backend to be absent\\"\\n        )\\n\\n    if mismatches:\\n        raise RuntimeError(\\n            \\"Incompatible DPO training environment (package mismatch: \\"\\n            + \\", \\".join(mismatches)\\n            + \\"). Install:\\\\n  \\"\\n            + _INSTALL_COMMAND\\n        )\\n    return versions\\n\\n\\ndef _verify_dpo_config_interface(dpo_config: type) -> None:\\n    parameters = set(inspect.signature(dpo_config).parameters)\\n    missing_parameters = sorted(_REQUIRED_DPO_PARAMETERS - parameters)\\n    if missing_parameters:\\n        raise RuntimeError(\\n            \\"Incompatible DPO training environment (DPOConfig lacks: \\"\\n            + \\", \\".join(missing_parameters)\\n            + \\"). Install:\\\\n  \\"\\n            + _INSTALL_COMMAND\\n        )\\n\\n\\ndef verify_training_environment() -> dict[str, str]:\\n    \\"\\"\\"Verify the package tuple and TRL interface before allocating a model.\\"\\"\\"\\n    versions = _verify_training_packages()\\n    try:\\n        from trl import DPOConfig\\n    except Exception as exc:\\n        raise RuntimeError(\\n            \\"Cannot import TRL\'s DPOConfig. Install the pinned training tuple with:\\\\n\\"\\n            f\\"  {_INSTALL_COMMAND}\\"\\n        ) from exc\\n\\n    _verify_dpo_config_interface(DPOConfig)\\n    logger.info(\\"Training packages: %s\\", \\", \\".join(f\\"{k}={v}\\" for k, v in versions.items()))\\n    return versions\\n\\n\\ndef load_pairs(path: str | Path) -> list[dict[str, Any]]:\\n    \\"\\"\\"Load and validate one versioned preference-pair JSONL file.\\"\\"\\"\\n    DPO_OUTPUT_FORMAT_VERSION = 1\\n    from personalization.profiles import train_personas\\n\\n    pair_path = Path(path)\\n    if not pair_path.is_file():\\n        raise FileNotFoundError(f\\"Preference-pair file not found: {pair_path}\\")\\n\\n    known_personas = {persona.id for persona in train_personas()}\\n    pairs: list[dict[str, Any]] = []\\n    seen_rows: dict[str, int] = {}\\n    for line_number, line in enumerate(pair_path.read_text(encoding=\\"utf-8\\").splitlines(), 1):\\n        if not line.strip():\\n            continue\\n        try:\\n            record = json.loads(line)\\n        except json.JSONDecodeError as exc:\\n            raise ValueError(f\\"{pair_path} line {line_number} is invalid JSON: {exc}\\") from exc\\n        if not isinstance(record, dict):\\n            raise ValueError(f\\"{pair_path} line {line_number} must contain a JSON object\\")\\n\\n        version = record.get(\\"format_version\\")\\n        if version != DPO_OUTPUT_FORMAT_VERSION:\\n            raise ValueError(\\n                f\\"{pair_path} line {line_number} has format version {version!r}, expected \\"\\n                f\\"{DPO_OUTPUT_FORMAT_VERSION}; regenerate with src/data/gen_dpo_data.py\\"\\n            )\\n        for field in _REQUIRED_FIELDS:\\n            value = record.get(field)\\n            if not isinstance(value, str) or not value.strip():\\n                raise ValueError(\\n                    f\\"{pair_path} line {line_number} field {field!r} must be a nonempty string\\"\\n                )\\n        if record[\\"persona_id\\"] not in known_personas:\\n            raise ValueError(\\n                f\\"{pair_path} line {line_number} has unknown training persona \\"\\n                f\\"{record[\'persona_id\']!r}; expected one of {sorted(known_personas)}\\"\\n            )\\n        if record[\\"chosen\\"].strip() == record[\\"rejected\\"].strip():\\n            raise ValueError(\\n                f\\"{pair_path} line {line_number} has identical chosen and rejected completions\\"\\n            )\\n\\n        row_key = json.dumps(record, ensure_ascii=False, sort_keys=True, separators=(\\",\\", \\":\\"))\\n        if row_key in seen_rows:\\n            raise ValueError(\\n                f\\"{pair_path} line {line_number} duplicates line {seen_rows[row_key]} exactly\\"\\n            )\\n        seen_rows[row_key] = line_number\\n        pairs.append(record)\\n\\n    if not pairs:\\n        raise ValueError(f\\"Preference-pair file is empty: {pair_path}\\")\\n    return pairs\\n\\n\\ndef _split_summary(pairs: list[dict[str, Any]]) -> dict[str, Any]:\\n    return {\\n        \\"rows\\": len(pairs),\\n        \\"questions\\": len({pair[\\"question_ref\\"] for pair in pairs}),\\n        \\"question_persona_keys\\": len(\\n            {(pair[\\"question_ref\\"], pair[\\"persona_id\\"]) for pair in pairs}\\n        ),\\n        \\"personas\\": dict(sorted(Counter(pair[\\"persona_id\\"] for pair in pairs).items())),\\n    }\\n\\n\\ndef validate_splits(\\n    train_pairs: list[dict[str, Any]], val_pairs: list[dict[str, Any]]\\n) -> dict[str, Any]:\\n    \\"\\"\\"Reject question leakage and return the split report stored in the manifest.\\"\\"\\"\\n    train_questions = {pair[\\"question_ref\\"] for pair in train_pairs}\\n    val_questions = {pair[\\"question_ref\\"] for pair in val_pairs}\\n    overlap = sorted(train_questions & val_questions)\\n    if overlap:\\n        sample = \\", \\".join(overlap[:5])\\n        raise ValueError(\\n            f\\"Train/validation question_ref overlap: {len(overlap)} questions; first: {sample}\\"\\n        )\\n    return {\\n        \\"train\\": _split_summary(train_pairs),\\n        \\"validation\\": _split_summary(val_pairs),\\n        \\"question_overlap\\": 0,\\n        \\"score_analysis\\": \\"unavailable: pair files do not store judge scores\\",\\n        \\"pair_type_analysis\\": \\"unavailable: pair files do not store pair provenance\\",\\n    }\\n\\n\\ndef _render_pair(pair: dict[str, Any], tokenizer: PreTrainedTokenizerBase) -> dict[str, str]:\\n    from personalization.profiles import render_profile\\n    from rag.rewriter import build_rewrite_messages\\n\\n    messages = build_rewrite_messages(render_profile(pair[\\"persona_id\\"]), pair[\\"query\\"])\\n    prompt = tokenizer.apply_chat_template(\\n        messages,\\n        tokenize=False,\\n        add_generation_prompt=True,\\n        enable_thinking=False,\\n    )\\n    return {\\"prompt\\": prompt, \\"chosen\\": pair[\\"chosen\\"], \\"rejected\\": pair[\\"rejected\\"]}\\n\\n\\ndef pairs_to_dataset(pairs: list[dict[str, Any]], tokenizer: PreTrainedTokenizerBase):\\n    \\"\\"\\"Convert validated rows to the explicit-prompt schema expected by DPOTrainer.\\"\\"\\"\\n    from datasets import Dataset\\n\\n    return Dataset.from_list([_render_pair(pair, tokenizer) for pair in pairs])\\n\\n\\ndef _percentile_nearest_rank(values: list[int], percentile: float) -> int:\\n    ordered = sorted(values)\\n    index = max(0, math.ceil(percentile * len(ordered)) - 1)\\n    return ordered[index]\\n\\n\\ndef _length_summary(values: list[int], limit: int) -> dict[str, int | float]:\\n    return {\\n        \\"min\\": min(values),\\n        \\"median\\": statistics.median(values),\\n        \\"p95\\": _percentile_nearest_rank(values, 0.95),\\n        \\"max\\": max(values),\\n        \\"limit\\": limit,\\n        \\"over_limit\\": sum(value > limit for value in values),\\n    }\\n\\n\\ndef _completion_token_length(text: str, tokenizer: PreTrainedTokenizerBase) -> int:\\n    token_ids = tokenizer(text, add_special_tokens=False)[\\"input_ids\\"]\\n    eos_token_id = tokenizer.eos_token_id\\n    if eos_token_id is None or not token_ids or token_ids[-1] != eos_token_id:\\n        return len(token_ids) + 1\\n    return len(token_ids)\\n\\n\\ndef measure_token_lengths(\\n    dataset: Any,\\n    tokenizer: PreTrainedTokenizerBase,\\n    training_config: dict[str, Any],\\n    *,\\n    split_name: str,\\n) -> dict[str, Any]:\\n    \\"\\"\\"Measure TRL\'s prompt/completion/full lengths and reject every truncation.\\"\\"\\"\\n    prompt_lengths: list[int] = []\\n    chosen_lengths: list[int] = []\\n    rejected_lengths: list[int] = []\\n    chosen_full_lengths: list[int] = []\\n    rejected_full_lengths: list[int] = []\\n\\n    for row in dataset:\\n        prompt_length = len(tokenizer(row[\\"prompt\\"], add_special_tokens=False)[\\"input_ids\\"])\\n        chosen_length = _completion_token_length(row[\\"chosen\\"], tokenizer)\\n        rejected_length = _completion_token_length(row[\\"rejected\\"], tokenizer)\\n        prompt_lengths.append(prompt_length)\\n        chosen_lengths.append(chosen_length)\\n        rejected_lengths.append(rejected_length)\\n        chosen_full_lengths.append(prompt_length + chosen_length)\\n        rejected_full_lengths.append(prompt_length + rejected_length)\\n\\n    prompt_limit = int(training_config[\\"max_prompt_length\\"])\\n    completion_limit = int(training_config[\\"max_completion_length\\"])\\n    full_limit = int(training_config[\\"max_length\\"])\\n    summary = {\\n        \\"prompt\\": _length_summary(prompt_lengths, prompt_limit),\\n        \\"chosen_completion_with_eos\\": _length_summary(chosen_lengths, completion_limit),\\n        \\"rejected_completion_with_eos\\": _length_summary(rejected_lengths, completion_limit),\\n        \\"prompt_plus_chosen\\": _length_summary(chosen_full_lengths, full_limit),\\n        \\"prompt_plus_rejected\\": _length_summary(rejected_full_lengths, full_limit),\\n    }\\n    violations = sum(int(stats[\\"over_limit\\"]) for stats in summary.values())\\n    zero_completions = sum(length <= 1 for length in chosen_lengths + rejected_lengths)\\n    if zero_completions:\\n        raise ValueError(f\\"{split_name} has {zero_completions} zero-token completions\\")\\n    if violations:\\n        details = \\", \\".join(\\n            f\\"{name}={stats[\'over_limit\']}\\" for name, stats in summary.items() if stats[\\"over_limit\\"]\\n        )\\n        raise ValueError(f\\"{split_name} exceeds configured token budgets: {details}\\")\\n    logger.info(\\"%s token lengths: %s\\", split_name, json.dumps(summary, sort_keys=True))\\n    return {\\"rows\\": len(dataset), \\"violations\\": 0, **summary}\\n\\n\\ndef resolve_arm(config: dict[str, Any], arm: str | None) -> tuple[str, dict[str, Any]]:\\n    \\"\\"\\"Resolve one objective and reject config changes that alter its fixed definition.\\"\\"\\"\\n    selected = arm or config.get(\\"arm\\") or \\"dpo\\"\\n    if selected not in _ARM_CONTRACT:\\n        raise ValueError(f\\"Unknown arm {selected!r}; expected one of {sorted(_ARM_CONTRACT)}\\")\\n    configured_arms = config.get(\\"arms\\")\\n    if not isinstance(configured_arms, dict):\\n        raise ValueError(\\"Config must define the fixed arms mapping\\")\\n    configured = configured_arms.get(selected)\\n    expected = _ARM_CONTRACT[selected]\\n    if configured != expected:\\n        raise ValueError(\\n            f\\"Config for arm {selected!r} contradicts the experiment contract: \\"\\n            f\\"expected {expected}, got {configured}\\"\\n        )\\n    return selected, dict(expected)\\n\\n\\ndef _derive_batch(training_config: dict[str, Any], world_size: int) -> tuple[int, int]:\\n    per_device = int(training_config[\\"per_device_train_batch_size\\"])\\n    target = int(training_config[\\"target_global_batch_size\\"])\\n    denominator = world_size * per_device\\n    if denominator <= 0 or target <= 0 or target % denominator:\\n        raise ValueError(\\n            \\"training.target_global_batch_size must be a positive multiple of \\"\\n            f\\"world_size * per_device_train_batch_size ({world_size} * {per_device})\\"\\n        )\\n    accumulation = target // denominator\\n    if accumulation < 1:\\n        raise ValueError(\\"Derived gradient accumulation must be at least 1\\")\\n    return accumulation, per_device * world_size * accumulation\\n\\n\\ndef _load_tokenizer(model_name: str, max_length: int):\\n    from transformers import AutoTokenizer\\n\\n    tokenizer = AutoTokenizer.from_pretrained(model_name, model_max_length=max_length)\\n    if tokenizer.pad_token is None:\\n        tokenizer.pad_token = tokenizer.eos_token\\n    return tokenizer\\n\\n\\ndef _prepare_data(config: dict[str, Any], tokenizer: PreTrainedTokenizerBase):\\n    data_config = config.get(\\"data\\", {})\\n    if not data_config.get(\\"val_path\\"):\\n        raise ValueError(\\n            \\"data.val_path is required; a row-level fallback would leak repeated questions\\"\\n        )\\n    train_path = Path(data_config[\\"train_path\\"])\\n    val_path = Path(data_config[\\"val_path\\"])\\n    train_pairs = load_pairs(train_path)\\n    val_pairs = load_pairs(val_path)\\n    split_summary = validate_splits(train_pairs, val_pairs)\\n    train_dataset = pairs_to_dataset(train_pairs, tokenizer)\\n    val_dataset = pairs_to_dataset(val_pairs, tokenizer)\\n    token_summary = {\\n        \\"train\\": measure_token_lengths(\\n            train_dataset, tokenizer, config[\\"training\\"], split_name=\\"train\\"\\n        ),\\n        \\"validation\\": measure_token_lengths(\\n            val_dataset, tokenizer, config[\\"training\\"], split_name=\\"validation\\"\\n        ),\\n    }\\n    return train_dataset, val_dataset, split_summary, token_summary, train_path, val_path\\n\\n\\ndef _seed_everything(seed: int) -> None:\\n    import numpy as np\\n    import torch\\n\\n    random.seed(seed)\\n    np.random.seed(seed)\\n    torch.manual_seed(seed)\\n    if torch.cuda.is_available():\\n        torch.cuda.manual_seed_all(seed)\\n\\n\\ndef _run_manifest(\\n    *,\\n    config: dict[str, Any],\\n    arm: str,\\n    arm_config: dict[str, Any],\\n    seed: int,\\n    package_versions: dict[str, str],\\n    world_size: int,\\n    accumulation: int,\\n    effective_batch: int,\\n    split_summary: dict[str, Any],\\n    token_summary: dict[str, Any],\\n    train_path: Path,\\n    val_path: Path,\\n) -> dict[str, Any]:\\n    return {\\n        \\"config\\": config,\\n        \\"arm\\": arm,\\n        \\"arm_config\\": arm_config,\\n        \\"seed\\": seed,\\n        \\"package_versions\\": package_versions,\\n        \\"world_size\\": world_size,\\n        \\"gradient_accumulation_steps\\": accumulation,\\n        \\"effective_global_batch_size\\": effective_batch,\\n        \\"splits\\": split_summary,\\n        \\"token_lengths\\": token_summary,\\n        \\"data_sha256\\": {\\"train\\": _sha256(train_path), \\"validation\\": _sha256(val_path)},\\n    }\\n\\n\\ndef run_preflight(config: dict[str, Any], *, arm: str | None = None, seed: int | None = None):\\n    \\"\\"\\"Validate environment, objective, data, and token budgets without loading the model.\\"\\"\\"\\n    package_versions = verify_training_environment()\\n    selected_arm, arm_config = resolve_arm(config, arm)\\n    selected_seed = int(config.get(\\"seed\\", 42) if seed is None else seed)\\n    world_size = int(os.environ.get(\\"WORLD_SIZE\\", \\"1\\"))\\n    accumulation, effective_batch = _derive_batch(config[\\"training\\"], world_size)\\n    tokenizer = _load_tokenizer(config[\\"model\\"][\\"name\\"], config[\\"model\\"][\\"max_seq_length\\"])\\n    _, _, split_summary, token_summary, train_path, val_path = _prepare_data(config, tokenizer)\\n    report = _run_manifest(\\n        config=config,\\n        arm=selected_arm,\\n        arm_config=arm_config,\\n        seed=selected_seed,\\n        package_versions=package_versions,\\n        world_size=world_size,\\n        accumulation=accumulation,\\n        effective_batch=effective_batch,\\n        split_summary=split_summary,\\n        token_summary=token_summary,\\n        train_path=train_path,\\n        val_path=val_path,\\n    )\\n    print(json.dumps(report, ensure_ascii=False, indent=2, sort_keys=True))\\n    return report\\n\\n\\ndef train(\\n    config: dict[str, Any],\\n    *,\\n    arm: str | None = None,\\n    seed: int | None = None,\\n    max_steps: int | None = None,\\n    resume_from_checkpoint: str | None = None,\\n) -> Path:\\n    \\"\\"\\"Train one preference arm and return its run directory.\\"\\"\\"\\n    package_versions = _verify_training_packages()\\n\\n    # Unsloth must patch Transformers, TRL, and PEFT before any of them are imported.\\n    from unsloth import FastLanguageModel  # noqa: I001\\n\\n    import torch\\n    from trl import DPOConfig, DPOTrainer\\n\\n    _verify_dpo_config_interface(DPOConfig)\\n    logger.info(\\n        \\"Training packages: %s\\",\\n        \\", \\".join(f\\"{k}={v}\\" for k, v in package_versions.items()),\\n    )\\n    selected_arm, arm_config = resolve_arm(config, arm)\\n    selected_seed = int(config.get(\\"seed\\", 42) if seed is None else seed)\\n    training_config = config[\\"training\\"]\\n    world_size = int(os.environ.get(\\"WORLD_SIZE\\", \\"1\\"))\\n    local_rank = int(os.environ.get(\\"LOCAL_RANK\\", \\"0\\"))\\n    if not torch.cuda.is_available():\\n        raise RuntimeError(\\"DPO training requires a CUDA GPU; use --preflight-only on CPU\\")\\n    torch.cuda.set_device(local_rank)\\n    _seed_everything(selected_seed)\\n\\n    accumulation, effective_batch = _derive_batch(training_config, world_size)\\n    run_dir = Path(training_config[\\"run_root\\"]) / selected_arm / f\\"seed-{selected_seed}\\"\\n    if _is_rank_zero() and run_dir.exists() and any(run_dir.iterdir()) and not resume_from_checkpoint:\\n        raise FileExistsError(\\n            f\\"Run directory is not empty: {run_dir}. Pass --resume-from-checkpoint explicitly.\\"\\n        )\\n    if resume_from_checkpoint and not Path(resume_from_checkpoint).is_dir():\\n        raise FileNotFoundError(f\\"Resume checkpoint not found: {resume_from_checkpoint}\\")\\n\\n    model_config = config[\\"model\\"]\\n    model, tokenizer = FastLanguageModel.from_pretrained(\\n        model_name=model_config[\\"name\\"],\\n        max_seq_length=int(model_config[\\"max_seq_length\\"]),\\n        dtype=None,\\n        load_in_4bit=bool(model_config[\\"load_in_4bit\\"]),\\n        # One replica per rank. Unsloth\'s default planner would shard a single-process run\\n        # across every visible GPU, which DDP replication and generation both reject.\\n        device_map={\\"\\": local_rank},\\n    )\\n    if tokenizer.pad_token is None:\\n        tokenizer.pad_token = tokenizer.eos_token\\n\\n    train_dataset, val_dataset, split_summary, token_summary, train_path, val_path = _prepare_data(\\n        config, tokenizer\\n    )\\n    lora_config = config[\\"lora\\"]\\n    model = FastLanguageModel.get_peft_model(\\n        model,\\n        r=int(lora_config[\\"r\\"]),\\n        lora_alpha=int(lora_config[\\"alpha\\"]),\\n        lora_dropout=float(lora_config[\\"dropout\\"]),\\n        target_modules=list(lora_config[\\"target_modules\\"]),\\n        bias=lora_config[\\"bias\\"],\\n        use_gradient_checkpointing=\\"unsloth\\",\\n        random_state=selected_seed,\\n    )\\n\\n    manifest = _run_manifest(\\n        config=config,\\n        arm=selected_arm,\\n        arm_config=arm_config,\\n        seed=selected_seed,\\n        package_versions=package_versions,\\n        world_size=world_size,\\n        accumulation=accumulation,\\n        effective_batch=effective_batch,\\n        split_summary=split_summary,\\n        token_summary=token_summary,\\n        train_path=train_path,\\n        val_path=val_path,\\n    )\\n    if _is_rank_zero():\\n        run_dir.mkdir(parents=True, exist_ok=True)\\n        _write_json(run_dir / \\"run_manifest.json\\", manifest)\\n\\n    configured_max_steps = -1 if max_steps is None else int(max_steps)\\n    if configured_max_steps == 0 or configured_max_steps < -1:\\n        raise ValueError(\\"max_steps must be a positive integer when supplied\\")\\n    args = DPOConfig(\\n        output_dir=str(run_dir),\\n        num_train_epochs=float(training_config[\\"epochs\\"]),\\n        max_steps=configured_max_steps,\\n        per_device_train_batch_size=int(training_config[\\"per_device_train_batch_size\\"]),\\n        per_device_eval_batch_size=int(training_config[\\"per_device_eval_batch_size\\"]),\\n        gradient_accumulation_steps=accumulation,\\n        learning_rate=float(training_config[\\"learning_rate\\"]),\\n        lr_scheduler_type=training_config[\\"lr_scheduler_type\\"],\\n        warmup_ratio=float(training_config[\\"warmup_ratio\\"]),\\n        max_grad_norm=float(training_config[\\"max_grad_norm\\"]),\\n        optim=training_config[\\"optim\\"],\\n        fp16=bool(training_config[\\"fp16\\"]),\\n        bf16=False,\\n        gradient_checkpointing=bool(training_config[\\"gradient_checkpointing\\"]),\\n        beta=float(training_config[\\"beta\\"]),\\n        loss_type=arm_config[\\"loss_type\\"],\\n        use_weighting=arm_config[\\"use_weighting\\"],\\n        label_smoothing=float(arm_config[\\"label_smoothing\\"]),\\n        rpo_alpha=float(arm_config[\\"rpo_alpha\\"]),\\n        max_prompt_length=int(training_config[\\"max_prompt_length\\"]),\\n        max_completion_length=int(training_config[\\"max_completion_length\\"]),\\n        max_length=int(training_config[\\"max_length\\"]),\\n        truncation_mode=training_config[\\"truncation_mode\\"],\\n        precompute_ref_log_probs=bool(training_config[\\"precompute_ref_log_probs\\"]),\\n        eval_strategy=training_config[\\"eval_strategy\\"],\\n        save_strategy=training_config[\\"save_strategy\\"],\\n        save_total_limit=int(training_config[\\"save_total_limit\\"]),\\n        load_best_model_at_end=bool(training_config[\\"load_best_model_at_end\\"]),\\n        metric_for_best_model=training_config[\\"metric_for_best_model\\"],\\n        greater_is_better=bool(training_config[\\"greater_is_better\\"]),\\n        logging_steps=int(training_config[\\"logging_steps\\"]),\\n        report_to=training_config[\\"report_to\\"],\\n        ddp_find_unused_parameters=bool(training_config[\\"ddp_find_unused_parameters\\"]),\\n        seed=selected_seed,\\n        data_seed=selected_seed,\\n    )\\n    logger.info(\\n        \\"Arm=%s loss=%s weighting=%s smoothing=%s rpo_alpha=%s world=%d \\"\\n        \\"accumulation=%d global_batch=%d\\",\\n        selected_arm,\\n        arm_config[\\"loss_type\\"],\\n        arm_config[\\"use_weighting\\"],\\n        arm_config[\\"label_smoothing\\"],\\n        arm_config[\\"rpo_alpha\\"],\\n        world_size,\\n        accumulation,\\n        effective_batch,\\n    )\\n    trainer = DPOTrainer(\\n        model=model,\\n        ref_model=None,\\n        args=args,\\n        train_dataset=train_dataset,\\n        eval_dataset=val_dataset,\\n        processing_class=tokenizer,\\n    )\\n    train_result = trainer.train(resume_from_checkpoint=resume_from_checkpoint)\\n    best_dir = run_dir / \\"dpo_best\\"\\n    trainer.save_model(str(best_dir))\\n    if _is_rank_zero():\\n        tokenizer.save_pretrained(best_dir)\\n        metrics = {\\n            \\"train\\": train_result.metrics,\\n            \\"log_history\\": trainer.state.log_history,\\n            \\"best_model_checkpoint\\": trainer.state.best_model_checkpoint,\\n            \\"best_metric\\": trainer.state.best_metric,\\n        }\\n        _write_json(run_dir / \\"training_metrics.json\\", metrics)\\n        manifest[\\"result\\"] = {\\n            \\"best_model_checkpoint\\": trainer.state.best_model_checkpoint,\\n            \\"best_metric\\": trainer.state.best_metric,\\n            \\"promoted_adapter\\": str(best_dir),\\n        }\\n        _write_json(run_dir / \\"run_manifest.json\\", manifest)\\n        logger.info(\\"Training complete: %s\\", best_dir)\\n    return run_dir\\n\\n\\ndef _parse_args() -> argparse.Namespace:\\n    parser = argparse.ArgumentParser(description=\\"Train the DPO-family query rewriter\\")\\n    parser.add_argument(\\"--config\\", type=Path, required=True, help=\\"Path to train_dpo YAML\\")\\n    parser.add_argument(\\"--arm\\", choices=sorted(_ARM_CONTRACT))\\n    parser.add_argument(\\"--seed\\", type=int)\\n    parser.add_argument(\\"--max-steps\\", type=int)\\n    parser.add_argument(\\"--resume-from-checkpoint\\")\\n    parser.add_argument(\\"--preflight-only\\", action=\\"store_true\\")\\n    return parser.parse_args()\\n\\n\\ndef main() -> None:\\n    logging.basicConfig(\\n        level=logging.INFO,\\n        format=\\"%(asctime)s [%(levelname)s] %(message)s\\",\\n        datefmt=\\"%Y-%m-%d %H:%M:%S\\",\\n    )\\n    args = _parse_args()\\n    config = yaml.safe_load(args.config.read_text(encoding=\\"utf-8\\"))\\n    if args.preflight_only:\\n        run_preflight(config, arm=args.arm, seed=args.seed)\\n        return\\n    train(\\n        config,\\n        arm=args.arm,\\n        seed=args.seed,\\n        max_steps=args.max_steps,\\n        resume_from_checkpoint=args.resume_from_checkpoint,\\n    )\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n"}')
for relative_path, content in SOURCE_FILES.items():
    path = WORKDIR / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
for package in ("rl", "rag", "personalization", "data"):
    (WORKDIR / "src" / package / "__init__.py").touch()

CONFIG_YAML = 'seed: 42\n\ndata:\n  train_path: data/dpo/train.jsonl\n  val_path: data/dpo/val.jsonl\n  questions_dir: data/questions\n\nmodel:\n  name: Qwen/Qwen3-4B\n  load_in_4bit: true\n  max_seq_length: 768\n\nlora:\n  r: 16\n  alpha: 32\n  dropout: 0.0\n  bias: none\n  target_modules:\n    - q_proj\n    - k_proj\n    - v_proj\n    - o_proj\n    - gate_proj\n    - up_proj\n    - down_proj\n\ntraining:\n  run_root: data/rl/dpo_rewriter_runs\n  # The seed-42 screening run peaked on validation preference accuracy after one epoch in\n  # every arm and declined afterwards while batch accuracy climbed past 0.9.\n  epochs: 1\n  target_global_batch_size: 8\n  per_device_train_batch_size: 1\n  per_device_eval_batch_size: 1\n  learning_rate: 1.0e-5\n  lr_scheduler_type: linear\n  warmup_ratio: 0.1\n  max_grad_norm: 1.0\n  optim: adamw_8bit\n  gradient_checkpointing: true\n  fp16: true\n  beta: 0.1\n  max_prompt_length: 576\n  max_completion_length: 224\n  max_length: 768\n  truncation_mode: keep_end\n  precompute_ref_log_probs: true\n  eval_strategy: epoch\n  save_strategy: epoch\n  save_total_limit: 3\n  load_best_model_at_end: true\n  # Not eval_loss: the robust objective is unbounded below and the WPO loss carries a\n  # policy-dependent weight, so lower loss can mean a less confident policy rather than a\n  # better one. Preference accuracy is bounded and comparable across epochs within an arm.\n  metric_for_best_model: eval_rewards/accuracies\n  greater_is_better: true\n  logging_steps: 10\n  report_to: none\n  ddp_find_unused_parameters: false\n\n# Fixed objective definitions. src/rl/dpo_train.py::resolve_arm rejects any edit here that\n# disagrees with its _ARM_CONTRACT. rpo_alpha adds a supervised NLL term on the chosen\n# rewrite, which counters the likelihood displacement that made every seed-42 arm push the\n# chosen log-probability down. TRL multiplies the anchor by the WPO weight in the wpo arm,\n# so the same alpha anchors that arm more weakly.\narms:\n  dpo:\n    loss_type: sigmoid\n    use_weighting: false\n    label_smoothing: 0.0\n    rpo_alpha: 1.0\n  wpo:\n    loss_type: sigmoid\n    use_weighting: true\n    label_smoothing: 0.0\n    rpo_alpha: 1.0\n  robust_dpo:\n    loss_type: robust\n    use_weighting: false\n    label_smoothing: 0.1\n    rpo_alpha: 1.0\n\ngeneration:\n  max_new_tokens: 224\n  do_sample: false\n  batch_size: 4\n  grok_model: grok-4-1-fast\n  grok_temperature: 0.0\n  output_root: data/rl/dpo_rewriter_outputs\n\ncomparison:\n  output_root: data/rl/dpo_rewriter_comparisons\n  screening_seed: 42\n  replication_seeds: [43, 44]\n  bootstrap_samples: 5000\n  max_workers: 4\n  # Thinking tokens are drawn from this same budget, and Gemini 3.x cannot switch thinking\n  # off, so this has to cover a minimal thought plus the JSON verdict.\n  judge_max_tokens: 512\n  # One of minimal, low, medium, high; empty lets the model pick its own default. Gemini\n  # 3.5 and newer reject the older thinking_budget field, so this is the only control.\n  judge_thinking_level: minimal\n  retry:\n    max_attempts: 3\n    initial_backoff_seconds: 1.0\n    backoff_multiplier: 2.0\n    max_backoff_seconds: 8.0\n'
CONFIG_PATH = WORKDIR / "train_dpo_embedded.yaml"
CONFIG_PATH.write_text(CONFIG_YAML, encoding="utf-8")
print(f"Wrote {len(SOURCE_FILES)} source files and {CONFIG_PATH}")


In [ ]:
import yaml

base_config = yaml.safe_load(CONFIG_YAML)
base_config["data"]["train_path"] = str(DATA_ROOT / "dpo" / "train.jsonl")
base_config["data"]["val_path"] = str(DATA_ROOT / "dpo" / "val.jsonl")
base_config["data"]["questions_dir"] = str(DATA_ROOT / "questions")
base_config["training"]["run_root"] = str(OUTPUT_ROOT / "runs")
base_config["generation"]["output_root"] = str(OUTPUT_ROOT / "outputs")
base_config["comparison"]["output_root"] = str(OUTPUT_ROOT / "comparisons")

FULL_CONFIG_PATH = WORKDIR / "train_dpo_runtime.yaml"
FULL_CONFIG_PATH.write_text(yaml.safe_dump(base_config, sort_keys=False), encoding="utf-8")
smoke_config = yaml.safe_load(yaml.safe_dump(base_config))
smoke_config["training"]["run_root"] = str(OUTPUT_ROOT / "smoke_runs")
SMOKE_CONFIG_PATH = WORKDIR / "train_dpo_smoke.yaml"
SMOKE_CONFIG_PATH.write_text(yaml.safe_dump(smoke_config, sort_keys=False), encoding="utf-8")
print("Runtime paths applied. Behavioral values still come from the embedded YAML.")


## Health and exact data preflight

In [ ]:
import importlib.metadata
import subprocess
import sys

import torch

n_gpus = torch.cuda.device_count()
print("Visible GPUs:", n_gpus)
for index in range(n_gpus):
    print(f"  cuda:{index} {torch.cuda.get_device_name(index)}")
assert n_gpus == 2, f"This DDP notebook requires two visible T4-class GPUs, got {n_gpus}"

# google-genai belongs here too: the tournament runs hours after this cell, and a silently
# failed install would only surface once every arm had already trained.
for package in (
    "unsloth",
    "trl",
    "transformers",
    "datasets",
    "peft",
    "accelerate",
    "google-genai",
):
    print(f"{package}={importlib.metadata.version(package)}")
try:
    torchao_version = importlib.metadata.version("torchao")
except importlib.metadata.PackageNotFoundError:
    print("torchao=not installed")
else:
    raise RuntimeError(
        f"torchao=={torchao_version} is still installed; rerun the package setup cell"
    )
for key in (
    "REWRITER_BASE_URL",
    "REWRITER_API_KEY",
    "DPO_EVAL_BASE_URL",
    "DPO_EVAL_API_KEY",
    "DPO_EVAL_MODEL",
):
    print(f"{key} present: {bool(os.environ.get(key))}")
print("Expected screening calls: 272 prompts x 10 pairs = 2720")
print("Expected replication calls: 272 prompts x 5 pairs x 2 seeds = 2720")
print("Total judge calls: 5440")

preflight_env = os.environ.copy()
preflight_env.update({"WORLD_SIZE": str(n_gpus), "RANK": "0"})
subprocess.run(
    [
        sys.executable,
        "-m",
        "rl.dpo_train",
        "--config",
        str(FULL_CONFIG_PATH),
        "--arm",
        ARMS[0],
        "--seed",
        str(SEED),
        "--preflight-only",
    ],
    check=True,
    env=preflight_env,
)


## Two-optimizer-step DDP smoke, first arm only

In [ ]:
import subprocess

# The arms differ only in loss configuration, so one smoke proves the DDP launch path for
# all of them. Repeating it per arm would burn several minutes of a batch run for nothing.
smoke_command = [
    "torchrun",
    "--standalone",
    "--nproc_per_node=2",
    "--tee",
    "3",
    "--log-dir",
    str(OUTPUT_ROOT / "smoke_torchrun_logs"),
    "-m",
    "rl.dpo_train",
    "--config",
    str(SMOKE_CONFIG_PATH),
    "--arm",
    ARMS[0],
    "--seed",
    str(SEED),
    "--max-steps",
    "2",
]
subprocess.run(smoke_command, check=True, env=os.environ.copy())


## Full three-epoch DDP training

In [ ]:
import subprocess

STAGE_STATUS = {}


def record(stage, error=None):
    STAGE_STATUS[stage] = "ok" if error is None else f"failed: {error}"
    print(f"[{stage}] {STAGE_STATUS[stage]}")


for arm in ARMS:
    full_command = [
        "torchrun",
        "--standalone",
        "--nproc_per_node=2",
        "--tee",
        "3",
        "--log-dir",
        str(OUTPUT_ROOT / "torchrun_logs" / f"{arm}-seed-{SEED}"),
        "-m",
        "rl.dpo_train",
        "--config",
        str(FULL_CONFIG_PATH),
        "--arm",
        arm,
        "--seed",
        str(SEED),
    ]
    # An arm that dies must not cost the arms that already finished their three epochs.
    try:
        subprocess.run(full_command, check=True, env=os.environ.copy())
    except subprocess.CalledProcessError as error:
        record(f"train:{arm}", error)
    else:
        record(f"train:{arm}")


## Deterministic model-output generation

In [ ]:
run_root = Path(base_config["training"]["run_root"])
promoted = {arm: run_root / arm / f"seed-{SEED}" for arm in ARMS}
trained = [arm for arm in ARMS if (promoted[arm] / "dpo_best").is_dir()]
print("Arms with a promoted adapter:", trained)
if not trained:
    record("generate", "no promoted adapter to generate from")
else:
    generation_command = [
        sys.executable,
        str(WORKDIR / "benchmarks" / "compare_dpo_rewriters.py"),
        "--config",
        str(FULL_CONFIG_PATH),
        "--comparison-seed",
        str(SEED),
        "--generate-only",
    ]
    for arm in trained:
        generation_command.extend(["--run", f"{arm}={promoted[arm]}"])
    try:
        subprocess.run(generation_command, check=True, env=os.environ.copy())
    except subprocess.CalledProcessError as error:
        record("generate", error)
    else:
        record("generate")


## Gemini blind pairwise tournament

In [ ]:
missing = [arm for arm in ARMS if not (promoted[arm] / "dpo_best").is_dir()]
if missing:
    print("Skipping the tournament; these arms have no promoted adapter:", missing)
    record("tournament", f"missing arms {missing}")
elif STAGE_STATUS.get("generate") != "ok":
    print("Skipping the tournament; deterministic output generation did not succeed.")
    record("tournament", "generation incomplete")
else:
    if SEED == 42:
        selected_pairs = []  # all 10 pairs among the five candidates
    else:
        selected_pairs = [
            f"dpo:{WINNING_VARIANT}",
            "base_qwen:dpo",
            "dpo:grok",
            f"base_qwen:{WINNING_VARIANT}",
            f"{WINNING_VARIANT}:grok",
        ]
    tournament_command = [
        sys.executable,
        str(WORKDIR / "benchmarks" / "compare_dpo_rewriters.py"),
        "--config",
        str(FULL_CONFIG_PATH),
        "--comparison-seed",
        str(SEED),
    ]
    for arm in ARMS:
        tournament_command.extend(["--run", f"{arm}={promoted[arm]}"])
    for selected_pair in selected_pairs:
        tournament_command.extend(["--pair", selected_pair])
    try:
        subprocess.run(tournament_command, check=True, env=os.environ.copy())
    except subprocess.CalledProcessError as error:
        record("tournament", error)
    else:
        record("tournament")


## Export artifacts

In [ ]:
import shutil

archive = shutil.make_archive(
    str(Path("/kaggle/working") / f"dpo-seed-{SEED}"),
    "zip",
    root_dir=OUTPUT_ROOT,
)
print("Created:", archive)

status_path = OUTPUT_ROOT / "run_status.json"
status_path.write_text(json.dumps(STAGE_STATUS, indent=2, sort_keys=True), encoding="utf-8")
print("=" * 72)
for stage in sorted(STAGE_STATUS):
    print(f"{stage:20s} {STAGE_STATUS[stage]}")
print("=" * 72)
failed = sorted(stage for stage, state in STAGE_STATUS.items() if state != "ok")
# Kaggle discards the output of a failed batch version, so a partially successful run must
# exit cleanly and report through the table above and run_status.json. Only a run that
# produced nothing at all raises, since it has no artifact left to preserve.
if failed and len(failed) == len(STAGE_STATUS):
    raise RuntimeError(f"Every stage failed: {failed}")
if failed:
    print("INCOMPLETE RUN. Failed stages:", failed)
else:
    print("All stages succeeded.")
